In [0]:
# Data Browser Publish — normalized synth_* payload + authoritative real metadata.
import hashlib
import json
import math
import re
import traceback
from copy import deepcopy

from pyspark.sql import functions as F

try:
    import yaml
except ModuleNotFoundError:
    class _JsonYaml:
        """JSON is a YAML 1.2 subset; use it when PyYAML is unavailable."""

        @staticmethod
        def safe_dump(value, **_kwargs):
            return json.dumps(value, indent=2, default=str)

        @staticmethod
        def safe_load(value):
            return json.loads(value)

    yaml = _JsonYaml()


def _ensure_text_widget(name, default):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default)


def _ensure_dropdown_widget(name, default, choices):
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices)


for _name, _default in (
    ("dataset_name", "preborn"),
    ("source_kind", "unity_catalog"),
    ("real_source", "5_projects.preborn"),
    ("synthetic_source", "5_projects.preborn"),
    ("synthetic_prefix", "synth_"),
    (
        "publish_root",
        "abfss://discovery@databrowserdiscovery.dfs.core.windows.net/datasets",
    ),
    (
        "scratch_root",
        "/Volumes/8_dev/synth_preborn_m/artifacts/discovery_dryrun",
    ),
    ("suppression_threshold", "10"),
    ("seed", "20260716"),
    ("target_file_mb", "128"),
    ("run_id", ""),
    ("run_at", ""),
):
    _ensure_text_widget(_name, _default)
_ensure_dropdown_widget("stats_basis", "synthetic", ["synthetic", "real_suppressed"])
_ensure_dropdown_widget("dry_run", "true", ["true", "false"])
_ensure_dropdown_widget("production_approved", "false", ["true", "false"])

P = {
    name: dbutils.widgets.get(name)
    for name in (
        "dataset_name",
        "source_kind",
        "real_source",
        "synthetic_source",
        "synthetic_prefix",
        "publish_root",
        "scratch_root",
        "stats_basis",
    )
}
P["k"] = int(dbutils.widgets.get("suppression_threshold"))
P["seed"] = int(dbutils.widgets.get("seed"))
P["target_file_mb"] = int(dbutils.widgets.get("target_file_mb"))
P["dry_run"] = dbutils.widgets.get("dry_run").lower() == "true"
P["production_approved"] = dbutils.widgets.get("production_approved").lower() == "true"
run_id = dbutils.widgets.get("run_id") or spark.sql(
    "SELECT replace(uuid(), '-', '')"
).first()[0]
run_at = dbutils.widgets.get("run_at") or spark.sql(
    "SELECT date_format(current_timestamp(), \"yyyy-MM-dd'T'HH:mm:ssXXX\")"
).first()[0]

assert P["source_kind"] == "unity_catalog", "only unity_catalog is implemented in v1"
assert re.fullmatch(r"[a-z][a-z0-9_]*", P["dataset_name"]), "unsafe dataset_name"
assert P["stats_basis"] in ("synthetic", "real_suppressed")
assert P["k"] >= 1 and P["target_file_mb"] >= 32
assert len(P["real_source"].split(".")) == 2
assert len(P["synthetic_source"].split(".")) == 2
rcat, rsch = P["real_source"].split(".", 1)
scat, ssch = P["synthetic_source"].split(".", 1)
for _identifier in (rcat, rsch, scat, ssch):
    assert re.fullmatch(r"[A-Za-z0-9_]+", _identifier), f"unsafe UC identifier {_identifier!r}"

runtime_config = {}
for _key, _value in (
    ("spark.sql.parquet.outputTimestampType", "TIMESTAMP_MICROS"),
    ("spark.sql.parquet.compression.codec", "snappy"),
):
    try:
        spark.conf.set(_key, _value)
        runtime_config[_key] = "set"
    except Exception as _error:
        runtime_config[_key] = f"writer_option_fallback: {type(_error).__name__}"
root = (P["scratch_root"] if P["dry_run"] else P["publish_root"]).rstrip("/")
if P["dry_run"]:
    assert root.startswith(("dbfs:/tmp/", "/tmp/", "/Volumes/")), "unsafe scratch root"
else:
    assert P["production_approved"], "dry_run=false requires production_approved=true"
    assert root.startswith(
        "abfss://discovery@databrowserdiscovery.dfs.core.windows.net/datasets"
    ), "unsafe publish root"
staging = f"{root}/_staging/{P['dataset_name']}/{run_id}"
live = f"{root}/{P['dataset_name']}"
archive = f"{root}/_archive/{P['dataset_name']}/{run_id}"
failure_path = f"{root}/_failures/{P['dataset_name']}/{run_id}.json"
dbutils.fs.rm(staging, True)
dbutils.fs.mkdirs(f"{staging}/stats")
dbutils.fs.mkdirs(f"{root}/_failures/{P['dataset_name']}")
diagnostic_path = f"{staging}/diagnostic.json"


def _mark(phase, **detail):
    dbutils.fs.put(
        diagnostic_path,
        json.dumps({"phase": phase, "detail": detail}, default=str),
        True,
    )


def _record_failure(phase, **detail):
    dbutils.fs.put(
        failure_path,
        json.dumps(
            {"phase": phase, "detail": detail, "traceback": traceback.format_exc()},
            default=str,
        )
        + "\n",
        True,
    )


_mark("parameters_ready")
print("dry_run", P["dry_run"], "basis", P["stats_basis"], "staging", staging)

In [0]:
"""Pure, Spark-free logic for the Data Browser dataset publisher (PREBORN first).

Every disclosure-sensitive or correctness-sensitive transform (type normalization
planning, suppression, rounding, ig_risk fail-closed, stats shaping) lives here so it is
unit-tested. The notebook's Spark cells call these functions; a published count or a cast
never bypasses this module.
"""


def map_storage_type(source_type: str) -> str:
    """Map a Spark/UC type string to a supported published storage type."""
    t = source_type.strip().lower()
    base = t.split("(", 1)[0].split("<", 1)[0].strip()
    if base in {"tinyint", "smallint", "int", "integer", "bigint", "long"}:
        return "int64"
    if base in {"float", "double", "decimal", "numeric"}:
        return "float64"
    if base in {"string", "varchar", "char"}:
        return "string"
    if base in {"boolean", "bool"}:
        return "bool"
    if base in {"timestamp", "timestamp_ntz"}:
        return "timestamp"
    if base == "date":
        return "date"
    return "excluded"


# Widening integer casts (source already narrower/equal) cannot introduce nulls.
_INT_WIDTH = {
    "tinyint": 1,
    "smallint": 2,
    "int": 4,
    "integer": 4,
    "bigint": 8,
    "long": 8,
}


def _basename(type_name):
    return type_name.strip().lower().split("(", 1)[0].split("<", 1)[0].strip()


def plan_cast(column, real_type, synth_type):
    """Plan conversion of a synthetic column to its real logical type."""
    published = map_storage_type(real_type)
    plan = {
        "column": column,
        "published_type": published,
        "source_type": real_type,
        "check_null_introduction": False,
        "check_midnight_only": False,
    }
    if published == "excluded":
        plan["action"] = "exclude"
        return plan
    if _basename(real_type) == _basename(synth_type):
        plan["action"] = "noop"
        return plan

    plan["action"] = "cast"
    real_base, synth_base = _basename(real_type), _basename(synth_type)
    plan["cast_to"] = (
        "date"
        if real_base == "date"
        else "timestamp"
        if real_base in ("timestamp", "timestamp_ntz")
        else real_base
    )
    if real_base == "date" and synth_base in ("timestamp", "timestamp_ntz"):
        plan["check_midnight_only"] = True
        plan["check_null_introduction"] = True
    elif (
        real_base in _INT_WIDTH
        and synth_base in _INT_WIDTH
        and _INT_WIDTH[synth_base] <= _INT_WIDTH[real_base]
    ):
        plan["check_null_introduction"] = False
    else:
        plan["check_null_introduction"] = True
    return plan


def round_to_5(n) -> int:
    if n is None:
        return 0
    return int(5 * ((float(n) + 2.5) // 5))


def suppress_counts(counts: dict, k: int, top_n=None):
    """Suppress small categories and optionally fold values beyond ``top_n``."""
    surviving = sorted(
        ((value, count) for value, count in counts.items() if count >= k),
        key=lambda item: (-item[1], str(item[0])),
    )
    suppressed = sum(count for value, count in counts.items() if count < k)
    if top_n is not None and len(surviving) > top_n:
        suppressed += sum(count for _, count in surviving[top_n:])
        surviving = surviving[:top_n]
    return [
        (value, round_to_5(count)) for value, count in surviving
    ], round_to_5(suppressed)


VALID_IG_RISK = ("low", "medium", "high")


def resolve_ig_risk(col_cfg):
    """Return ``(risk, defaulted)`` and fail closed when classification is absent."""
    if not col_cfg or "ig_risk" not in col_cfg:
        return "high", True
    risk = col_cfg["ig_risk"]
    if risk not in VALID_IG_RISK:
        raise ValueError(
            f"invalid ig_risk {risk!r}; expected one of {VALID_IG_RISK}"
        )
    return risk, False


def _null_frac(null_count, row_count):
    return round(null_count / row_count, 4) if row_count else 0.0


def shape_column_stats(basis, ig_risk, published_type, agg, k, labels=None):
    """Shape pre-aggregated facts into publishable per-column statistics."""
    labels = labels or {}
    suppress = basis == "real"
    out = {
        "type": published_type,
        "ig_risk": ig_risk,
        "basis": basis,
        "null_frac": _null_frac(
            agg.get("null_count", 0), agg.get("row_count", 0)
        ),
    }

    if suppress and ig_risk == "high":
        if "distinct_count" in agg:
            out["distinct_count"] = round_to_5(agg["distinct_count"])
        return out

    if published_type in ("int64", "float64") and "mean" in agg:
        for key in ("mean", "std", "p5", "p25", "p50", "p75", "p95"):
            if key in agg:
                out[key] = agg[key]
        histogram = [
            (bucket, count)
            for bucket, count in agg.get("histogram", [])
            if count >= k or not suppress
        ]
        out["histogram"] = [
            {"bin": bucket, "count": round_to_5(count)}
            for bucket, count in histogram
        ]
        if "distinct_count" in agg:
            out["distinct_count"] = round_to_5(agg["distinct_count"])
        return out

    if published_type == "bool":
        out["true_count"] = round_to_5(agg.get("true_count", 0))
        out["false_count"] = round_to_5(agg.get("false_count", 0))
        return out

    if published_type in ("timestamp", "date"):
        out["min_month"] = agg.get("min_month")
        out["max_month"] = agg.get("max_month")
        month_histogram = [
            (month, count)
            for month, count in agg.get("month_histogram", [])
            if count >= k or not suppress
        ]
        out["month_histogram"] = [
            {"month": month, "count": round_to_5(count)}
            for month, count in month_histogram
        ]
        return out

    if agg.get("high_cardinality"):
        out["distinct_count"] = round_to_5(agg.get("distinct_count", 0))
        for key in ("length_min", "length_median", "length_max"):
            if agg.get(key) is not None:
                out[key] = agg[key]
        return out

    out["distinct_count"] = round_to_5(agg.get("distinct_count", 0))
    value_counts = agg.get("value_counts", {})
    if suppress:
        kept, other = suppress_counts(value_counts, k=k, top_n=50)
    else:
        sorted_values = sorted(
            value_counts.items(), key=lambda item: (-item[1], str(item[0]))
        )[:50]
        kept = [
            (value, round_to_5(count)) for value, count in sorted_values
        ]
        other = round_to_5(
            sum(value_counts.values()) - sum(count for _, count in sorted_values)
        )
    out["top_values"] = [
        {"value": value, "count": count, "label": labels[value]}
        if value in labels
        else {"value": value, "count": count}
        for value, count in kept
    ]
    out["suppressed_other_count"] = other
    return out


def round_to_nearest(n, base):
    return int(base * round(float(n) / base)) if n is not None else 0


def build_table_entry(table_name, table_cfg, cols_meta, synth_rows, source_rows):
    """Build one ``schema.yaml`` table entry from authoritative real metadata."""
    primary_key = table_cfg.get("pk")
    fk_by_column = {
        fk["column"]: fk["references"] for fk in table_cfg.get("fks", [])
    }
    column_configs = table_cfg.get("columns", {})
    excluded, columns = [], []
    for column in sorted(cols_meta, key=lambda item: item["ordinal"]):
        column_config = column_configs.get(column["name"], {})
        cast_plan = plan_cast(
            column["name"], column["real_type"], column["synth_type"]
        )
        published_type = cast_plan["published_type"]
        entry_column = {
            "name": column["name"],
            "type": published_type,
            "comment": column["comment"],
            "ig_risk": resolve_ig_risk(column_config)[0],
            "nullable": bool(column["nullable"]),
        }
        if (
            published_type != column["real_type"].strip().lower()
            or cast_plan["action"] == "cast"
        ):
            entry_column["source_type"] = column["real_type"]
        if published_type == "excluded":
            excluded.append(
                {
                    "table": table_name,
                    "column": column["name"],
                    "reason": f"unsupported type {column['real_type']}",
                }
            )
        if column["name"] == primary_key:
            entry_column["pk"] = True
        if column["name"] in fk_by_column:
            entry_column["fk"] = fk_by_column[column["name"]]
        for optional in (
            "code_system",
            "unit",
            "synthesis",
            "allowed_values",
            "semantic_role",
        ):
            if optional in column_config:
                entry_column[optional] = column_config[optional]
        columns.append(entry_column)

    table_entry = {
        "kind": table_cfg["kind"],
        "view_name": table_cfg["view_name"],
        "description": table_cfg.get("description", ""),
        "grain": table_cfg.get("grain"),
        "rows": synth_rows,
        "source_rows_rounded": round_to_nearest(source_rows, 1000),
        "columns": columns,
    }
    if table_cfg.get("pk_note"):
        table_entry["pk_note"] = table_cfg["pk_note"]
    if table_cfg.get("partition_by"):
        table_entry["partition_by"] = table_cfg["partition_by"]
    return table_entry, excluded


_VALID_CARDINALITY = (
    "one_to_one",
    "many_to_one",
    "one_to_many",
    "many_to_many",
)


def _endpoint_ok(tables, endpoint):
    if not isinstance(endpoint, str) or "." not in endpoint:
        return False
    table_name, column_name = endpoint.split(".", 1)
    return (
        table_name in tables
        and column_name in tables[table_name].get("columns", {})
    )



_VALID_INSIGHT_LENSES = ("research", "operational")
_VALID_COMPARISON_KINDS = ("numeric", "proportion")
_CAUSAL_LABEL_RE = re.compile(
    r"\b(caus(?:e|es|ed|ing|al)|effect(?:s|ed|ing)?|impact(?:s|ed|ing)?|"
    r"leads?\s+to|results?\s+in|due\s+to|attribut(?:e|es|ed|able))\b",
    re.IGNORECASE,
)


def _label_has_causal_wording(value):
    return isinstance(value, str) and bool(_CAUSAL_LABEL_RE.search(value))


def _contains_concept_zero(values):
    return any(value == 0 or str(value).strip() == "0" for value in values)


def validate_insights_config(cfg):
    """Validate the optional, curator-authored insights extension."""
    insights = cfg.get("insights")
    if not insights:
        return []

    errors = []
    tables = cfg.get("tables", {})
    if insights.get("default_lens") not in _VALID_INSIGHT_LENSES:
        errors.append(
            "insights.default_lens must be research|operational"
        )
    recipes = insights.get("comparisons")
    if not isinstance(recipes, list) or not recipes:
        return errors + ["insights.comparisons must be a non-empty curator list"]

    seen_ids = set()
    for index, recipe in enumerate(recipes):
        prefix = f"insights.comparisons[{index}]"
        if not isinstance(recipe, dict):
            errors.append(f"{prefix}: recipe must be an object")
            continue

        recipe_id = recipe.get("id")
        if not isinstance(recipe_id, str) or not re.fullmatch(
            r"[a-z][a-z0-9_]*", recipe_id
        ):
            errors.append(f"{prefix}: id must be lower_snake_case")
        elif recipe_id in seen_ids:
            errors.append(f"{prefix}: duplicate id {recipe_id}")
        else:
            seen_ids.add(recipe_id)

        kind = recipe.get("kind")
        if kind not in _VALID_COMPARISON_KINDS:
            errors.append(f"{prefix}: kind must be numeric|proportion")

        table_name = recipe.get("table")
        outcome = recipe.get("outcome")
        cohort_column = recipe.get("cohort_column")
        if table_name not in tables:
            errors.append(f"{prefix}: unknown table {table_name}")
            columns = {}
        else:
            columns = tables[table_name].get("columns", {})

        if not isinstance(outcome, str) or "." in outcome or outcome not in columns:
            errors.append(f"{prefix}: outcome must be a same-table configured column")
            outcome_config = {}
        else:
            outcome_config = columns[outcome]
        if (
            not isinstance(cohort_column, str)
            or "." in cohort_column
            or cohort_column not in columns
        ):
            errors.append(
                f"{prefix}: cohort_column must be a same-table configured column"
            )
            cohort_config = {}
        else:
            cohort_config = columns[cohort_column]

        if outcome_config.get("ig_risk", "high") == "high":
            errors.append(f"{prefix}: high-risk outcomes are forbidden")
        if outcome_config.get("semantic_role") in ("identifier", "free_text"):
            errors.append(f"{prefix}: identifier/free-text outcomes are forbidden")
        if cohort_config.get("ig_risk", "high") == "high":
            errors.append(f"{prefix}: high-risk cohort columns are forbidden")
        if kind == "numeric" and outcome_config.get("semantic_role") != "measure":
            errors.append(f"{prefix}: numeric outcome must have semantic_role=measure")
        if kind == "proportion" and outcome_config.get("semantic_role") not in (
            "category",
            "code",
        ):
            errors.append(
                f"{prefix}: proportion outcome must be a configured categorical column"
            )

        for forbidden in ("join", "joins", "sql", "where"):
            if forbidden in recipe:
                errors.append(
                    f"{prefix}: {forbidden} is forbidden; "
                    "cross-table/query recipes are not allowed"
                )

        groups = {}
        for group_name in ("reference", "comparison"):
            group = recipe.get(group_name)
            if not isinstance(group, dict):
                errors.append(f"{prefix}: missing {group_name} cohort")
                continue
            values = group.get("values")
            if not isinstance(values, list) or not values:
                errors.append(f"{prefix}: {group_name}.values must be non-empty")
                values = []
            if _contains_concept_zero(values):
                errors.append(f"{prefix}: concept 0 cannot appear in a cohort")
            groups[group_name] = {str(value) for value in values}
            label = group.get("label")
            if not isinstance(label, str) or not label.strip():
                errors.append(f"{prefix}: {group_name}.label is required")
            elif _label_has_causal_wording(label):
                errors.append(f"{prefix}: causal wording is forbidden in labels")

        if groups.get("reference", set()) & groups.get("comparison", set()):
            errors.append(f"{prefix}: reference and comparison cohorts overlap")

        recipe_label = recipe.get("label")
        if not isinstance(recipe_label, str) or not recipe_label.strip():
            errors.append(f"{prefix}: label is required")
        elif _label_has_causal_wording(recipe_label):
            errors.append(f"{prefix}: causal wording is forbidden in labels")

        if kind == "proportion":
            outcomes = {}
            for outcome_group in ("success", "failure"):
                group = recipe.get(outcome_group)
                if not isinstance(group, dict):
                    errors.append(f"{prefix}: proportion recipe needs {outcome_group}")
                    continue
                values = group.get("values")
                if not isinstance(values, list) or not values:
                    errors.append(
                        f"{prefix}: {outcome_group}.values must be non-empty"
                    )
                    values = []
                if _contains_concept_zero(values):
                    errors.append(
                        f"{prefix}: concept 0 cannot appear in binary outcome values"
                    )
                outcomes[outcome_group] = {str(value) for value in values}
                label = group.get("label")
                if not isinstance(label, str) or not label.strip():
                    errors.append(f"{prefix}: {outcome_group}.label is required")
                elif _label_has_causal_wording(label):
                    errors.append(f"{prefix}: causal wording is forbidden in labels")
            if outcomes.get("success", set()) & outcomes.get("failure", set()):
                errors.append(f"{prefix}: success and failure values overlap")

    return errors


def validate_config(cfg):
    """Return config validation errors; an empty list means the profile is valid."""
    errors = []
    tables = cfg.get("tables", {})
    if not tables:
        return ["config has no tables"]

    for table_name, table in tables.items():
        for required in ("kind", "view_name", "pk"):
            if not table.get(required):
                errors.append(f"table {table_name}: missing {required}")
        if table.get("kind") not in ("dimension", "fact", "reference"):
            errors.append(
                f"table {table_name}: kind must be dimension|fact|reference"
            )
        primary_key = table.get("pk")
        if primary_key and primary_key not in table.get("columns", {}):
            errors.append(
                f"table {table_name}: pk column unknown {table_name}.{primary_key}"
            )
        for foreign_key in table.get("fks", []):
            source_column = foreign_key.get("column", "")
            if source_column not in table.get("columns", {}):
                errors.append(
                    f"table {table_name}: fk source column unknown {source_column}"
                )
            reference = foreign_key.get("references", "")
            if not _endpoint_ok(tables, reference):
                errors.append(
                    f"table {table_name}: fk references unknown endpoint {reference}"
                )

    for relationship in cfg.get("relationships", []):
        name = relationship.get("name", "?")
        if not _endpoint_ok(tables, relationship.get("from", "")):
            errors.append(
                f"relationship {name}: from endpoint unknown {relationship.get('from')}"
            )
        if relationship.get("polymorphic"):
            targets = relationship.get("targets", [])
            if not targets:
                errors.append(f"relationship {name}: polymorphic targets missing")
            for target in targets:
                if not _endpoint_ok(tables, target):
                    errors.append(
                        f"relationship {name}: target endpoint unknown {target}"
                    )
        elif not _endpoint_ok(tables, relationship.get("to", "")):
            errors.append(
                f"relationship {name}: to endpoint unknown {relationship.get('to')}"
            )
        if relationship.get("cardinality") not in _VALID_CARDINALITY:
            errors.append(
                f"relationship {name}: bad cardinality {relationship.get('cardinality')}"
            )
    errors.extend(validate_insights_config(cfg))
    return errors


def build_dataset_block(
    name,
    description,
    version,
    published_at,
    seed,
    stats_basis,
    owner=None,
    source_system=None,
    refresh_cadence=None,
    insights=None,
):
    block = {
        "name": name,
        "description": description,
        "version": version,
        "published_at": published_at,
        "stats_basis": stats_basis,
        "synthesis": {
            "mode": "prebuilt",
            "seed": seed,
            "notes": (
                "Payload rows are an upstream synthetic copy, normalized to the real "
                "logical schema; not generated here."
            ),
        },
    }
    for key, value in (
        ("owner", owner),
        ("source_system", source_system),
        ("refresh_cadence", refresh_cadence),
    ):
        if value:
            block[key] = value
    if insights:
        block["insights"] = deepcopy(insights)
    return block


def build_receipt(
    dataset,
    run_at,
    run_id,
    notebook_version,
    params,
    source_snapshot,
    coercions,
    excluded_columns,
    untagged_high,
    validation,
):
    return {
        "dataset": dataset,
        "run_at": run_at,
        "run_id": run_id,
        "notebook_version": notebook_version,
        "parameters": params,
        "source_snapshot": source_snapshot,
        "type_coercions": coercions,
        "excluded_columns": excluded_columns,
        "untagged_columns_treated_high": untagged_high,
        "validation": validation,
    }


def check_plausibility(
    real_null_frac,
    synth_null_frac,
    real_median,
    synth_median,
    null_tol_pp=0.05,
    median_tol_frac=0.10,
):
    """Check the null-fraction and median tolerances from the package contract."""
    if abs(real_null_frac - synth_null_frac) > null_tol_pp:
        return (
            False,
            f"null fraction drift {real_null_frac:.3f}->{synth_null_frac:.3f} "
            f"exceeds {null_tol_pp}",
        )
    if real_median not in (None, 0):
        if synth_median is None:
            return False, "median drift cannot be evaluated: synthetic median is null"
        if abs(real_median - synth_median) / abs(real_median) > median_tol_frac:
            return (
                False,
                f"median drift {real_median}->{synth_median} exceeds "
                f"{median_tol_frac:.0%}",
            )
    return True, "ok"



INSIGHT_MIN_COHORT_N = 30
_Z_975 = 1.959963984540054


def _public_metric(value, digits):
    if value is None:
        return None
    value = float(value)
    if not math.isfinite(value):
        return None
    rounded = round(value, digits)
    return 0.0 if rounded == 0 else rounded


def _student_t_critical_975(degrees_of_freedom):
    """Accurate asymptotic t(0.975) expansion; comparisons require n>=30 per arm."""
    if not math.isfinite(degrees_of_freedom) or degrees_of_freedom > 1_000_000:
        return _Z_975
    inverse_df = 1.0 / float(degrees_of_freedom)
    z = _Z_975
    return (
        z
        + (z**3 + z) * inverse_df / 4.0
        + (5 * z**5 + 16 * z**3 + 3 * z) * inverse_df**2 / 96.0
        + (
            3 * z**7
            + 19 * z**5
            + 17 * z**3
            - 15 * z
        )
        * inverse_df**3
        / 384.0
    )


def welch_numeric_difference(reference, comparison):
    """Return comparison-reference Welch CI and Hedges' g."""
    n_reference = int(reference["n"])
    n_comparison = int(comparison["n"])
    mean_reference = float(reference["mean"])
    mean_comparison = float(comparison["mean"])
    sd_reference = float(reference["sd"] or 0.0)
    sd_comparison = float(comparison["sd"] or 0.0)
    difference = mean_comparison - mean_reference

    reference_term = sd_reference**2 / n_reference
    comparison_term = sd_comparison**2 / n_comparison
    standard_error_sq = reference_term + comparison_term
    if standard_error_sq > 0:
        denominator = (
            reference_term**2 / (n_reference - 1)
            + comparison_term**2 / (n_comparison - 1)
        )
        degrees_of_freedom = (
            standard_error_sq**2 / denominator
            if denominator > 0
            else float("inf")
        )
        half_width = _student_t_critical_975(degrees_of_freedom) * math.sqrt(
            standard_error_sq
        )
    else:
        half_width = 0.0

    pooled_df = n_reference + n_comparison - 2
    pooled_variance = (
        (n_reference - 1) * sd_reference**2
        + (n_comparison - 1) * sd_comparison**2
    ) / pooled_df
    if pooled_variance > 0:
        correction = 1.0 - 3.0 / (4.0 * pooled_df - 1.0)
        hedges_g = correction * difference / math.sqrt(pooled_variance)
    else:
        hedges_g = None

    return {
        "estimate": difference,
        "ci95": [difference - half_width, difference + half_width],
        "hedges_g": hedges_g,
    }


def _wilson_interval(successes, total):
    proportion = successes / total
    denominator = 1.0 + _Z_975**2 / total
    centre = (
        proportion + _Z_975**2 / (2.0 * total)
    ) / denominator
    half_width = (
        _Z_975
        * math.sqrt(
            proportion * (1.0 - proportion) / total
            + _Z_975**2 / (4.0 * total**2)
        )
        / denominator
    )
    return max(0.0, centre - half_width), min(1.0, centre + half_width)


def newcombe_proportion_difference(reference, comparison):
    """Return comparison-reference difference with Newcombe-Wilson 95% CI."""
    n_reference = int(reference["n"])
    n_comparison = int(comparison["n"])
    success_reference = int(reference["successes"])
    success_comparison = int(comparison["successes"])
    proportion_reference = success_reference / n_reference
    proportion_comparison = success_comparison / n_comparison
    reference_low, reference_high = _wilson_interval(
        success_reference, n_reference
    )
    comparison_low, comparison_high = _wilson_interval(
        success_comparison, n_comparison
    )
    difference = proportion_comparison - proportion_reference
    lower = difference - math.sqrt(
        (proportion_comparison - comparison_low) ** 2
        + (reference_high - proportion_reference) ** 2
    )
    upper = difference + math.sqrt(
        (comparison_high - proportion_comparison) ** 2
        + (proportion_reference - reference_low) ** 2
    )
    return {
        "estimate": difference,
        "ci95": [max(-1.0, lower), min(1.0, upper)],
    }


def _contains_forbidden_p_value(value):
    if isinstance(value, dict):
        for key, child in value.items():
            normalized = str(key).lower().replace("-", "_")
            if normalized in {"p", "pvalue", "p_value"}:
                return True
            if _contains_forbidden_p_value(child):
                return True
    elif isinstance(value, list):
        return any(_contains_forbidden_p_value(child) for child in value)
    return False


def comparison_document_errors(document, recipes):
    errors = []
    expected_ids = [recipe["id"] for recipe in recipes]
    results = document.get("results")
    if document.get("basis") != "real_suppressed":
        errors.append("basis must be real_suppressed")
    if not document.get("computed_at"):
        errors.append("computed_at is required")
    if not isinstance(results, list):
        return errors + ["results must be a list"]
    actual_ids = [result.get("id") for result in results]
    if actual_ids != expected_ids:
        errors.append(
            f"result ids/order differ: expected={expected_ids} actual={actual_ids}"
        )
    if len(actual_ids) != len(set(actual_ids)):
        errors.append("result ids are not unique")
    if _contains_forbidden_p_value(document):
        errors.append("p-values are forbidden")

    recipe_by_id = {recipe["id"]: recipe for recipe in recipes}
    for result in results:
        recipe = recipe_by_id.get(result.get("id"))
        if not recipe:
            continue
        prefix = result["id"]
        if result.get("kind") != recipe["kind"]:
            errors.append(f"{prefix}: kind mismatch")
        if result.get("table") != recipe["table"]:
            errors.append(f"{prefix}: table mismatch")
        if result.get("outcome") != recipe["outcome"]:
            errors.append(f"{prefix}: outcome mismatch")
        if not isinstance(result.get("sufficient"), bool):
            errors.append(f"{prefix}: sufficient must be boolean")

        for arm_name in ("reference", "comparison"):
            arm = result.get(arm_name, {})
            if arm.get("label") != recipe[arm_name]["label"]:
                errors.append(f"{prefix}: {arm_name} label mismatch")
            count = arm.get("n")
            if count is not None and (
                not isinstance(count, int) or count < 0 or count % 5 != 0
            ):
                errors.append(f"{prefix}: {arm_name} n is not rounded to 5")

        if result.get("sufficient"):
            if any(
                "n" not in result.get(arm_name, {})
                for arm_name in ("reference", "comparison")
            ):
                errors.append(f"{prefix}: sufficient result lacks cohort counts")
            if recipe["kind"] == "numeric":
                for arm_name in ("reference", "comparison"):
                    arm = result.get(arm_name, {})
                    if not all(key in arm for key in ("mean", "sd")):
                        errors.append(f"{prefix}: numeric {arm_name} summary incomplete")
                difference = result.get("difference", {})
                if not all(
                    key in difference for key in ("estimate", "ci95", "hedges_g")
                ):
                    errors.append(f"{prefix}: numeric difference incomplete")
            else:
                for arm_name in ("reference", "comparison"):
                    if "proportion" not in result.get(arm_name, {}):
                        errors.append(
                            f"{prefix}: proportion {arm_name} summary incomplete"
                        )
                difference = result.get("difference", {})
                if not all(key in difference for key in ("estimate", "ci95")):
                    errors.append(f"{prefix}: proportion difference incomplete")
        else:
            if "difference" in result:
                errors.append(f"{prefix}: insufficient result contains an estimate")
            for arm_name in ("reference", "comparison"):
                arm = result.get(arm_name, {})
                if any(key in arm for key in ("mean", "sd", "proportion")):
                    errors.append(
                        f"{prefix}: insufficient result contains arm estimates"
                    )
    return errors


In [0]:
"""Hand-authored Data Browser dataset profile for PREBORN (5_projects.preborn).

Source of record for keys/relationships/semantics: UC has NO PK/FK constraints or tags on
this schema (verified 2026-07-16). Declared from OMOP CDM v5.4 + PREBORN extensions.
Columns not listed default to ig_risk=high (fail-closed). provider/care_site/location are
absent in PREBORN and their synth fields are empty: retain columns but emit NO fk to them.
"""


def _fk(column, references):
    return {"column": column, "references": references}


_ID = {
    "ig_risk": "low",
    "synthesis": "faked",
    "semantic_role": "identifier",
}
_FK_ID = {
    "ig_risk": "low",
    "synthesis": "conditional",
    "semantic_role": "identifier",
}
_POLYMORPHIC_ID = {
    "ig_risk": "low",
    "synthesis": "conditional",
    "semantic_role": "identifier",
}
_CONCEPT = {
    "ig_risk": "low",
    "synthesis": "marginal",
    "code_system": "snomed",
    "semantic_role": "category",
}
_LOINC_CONCEPT = {
    "ig_risk": "low",
    "synthesis": "marginal",
    "code_system": "loinc",
    "semantic_role": "category",
}
_OMOP_CONCEPT = {
    "ig_risk": "low",
    "synthesis": "constant",
    "code_system": "omop",
    "semantic_role": "category",
}
_UNIT_CONCEPT = {
    "ig_risk": "low",
    "synthesis": "marginal",
    "code_system": "ucum",
    "semantic_role": "unit",
}
_DATE_HIGH = {
    "ig_risk": "high",
    "synthesis": "conditional",
    "semantic_role": "temporal",
}
_SITE = {
    "ig_risk": "low",
    "synthesis": "constant",
    "semantic_role": "category",
}
_SOURCE_TABLE = {
    "ig_risk": "low",
    "synthesis": "constant",
    "semantic_role": "category",
}
_REFERENCE_ID = {
    "ig_risk": "low",
    "synthesis": "constant",
    "semantic_role": "identifier",
}
_REFERENCE_CATEGORY = {
    "ig_risk": "low",
    "synthesis": "constant",
    "semantic_role": "category",
}
_REFERENCE_LABEL = {
    "ig_risk": "low",
    "synthesis": "constant",
    "semantic_role": "label",
}


DATASET_CONFIG = {
    "dataset_name": "preborn",
    "synthetic_prefix": "synth_",
    "description": (
        "PREBORN: an OMOP CDM v5.4 maternity/neonatal subset with maternity "
        "extension tables (pregnancy, infant) and a bundled vocabulary. Payload "
        "is a synthetic copy normalized to the real logical schema; metadata is "
        "from the real cohort."
    ),
    "owner": "PREBORN / maternity OMOP programme",
    "source_system": "5_projects.preborn (real) + synth_* (prebuilt synthetic payload)",
    "insights": {
        "default_lens": "research",
        "comparisons": [
            {
                "id": "gest_length_by_outcome",
                "label": "Gestational length: live birth vs stillbirth",
                "table": "pregnancy",
                "outcome": "gestational_length_in_day",
                "cohort_column": "pregnancy_outcome_concept_id",
                "reference": {
                    "values": [45883764],
                    "label": "Live birth",
                },
                "comparison": {
                    "values": [37017027, 45773593, 443213],
                    "label": "Stillbirth",
                },
                "kind": "numeric",
            },
            {
                "id": "birth_weight_by_outcome",
                "label": "Birth weight: live birth vs stillbirth",
                "table": "infant",
                "outcome": "birth_weight",
                "cohort_column": "birth_outcome_concept_id",
                "reference": {
                    "values": [45883764],
                    "label": "Live birth",
                },
                "comparison": {
                    "values": [37017027, 45773593, 443213],
                    "label": "Stillbirth",
                },
                "kind": "numeric",
            },
            {
                "id": "apgar_by_outcome",
                "label": "Apgar score: live birth vs stillbirth",
                "table": "infant",
                "outcome": "birth_apgar",
                "cohort_column": "birth_outcome_concept_id",
                "reference": {
                    "values": [45883764],
                    "label": "Live birth",
                },
                "comparison": {
                    "values": [37017027, 45773593, 443213],
                    "label": "Stillbirth",
                },
                "kind": "numeric",
            },
            {
                "id": "live_birth_by_delivery_mode",
                "label": "Live-birth proportion: spontaneous vaginal vs caesarean",
                "table": "pregnancy",
                "outcome": "pregnancy_outcome_concept_id",
                "cohort_column": "pregnancy_mode_delivery_concept_id",
                "reference": {
                    "values": [45885338],
                    "label": "Spontaneous vaginal",
                },
                "comparison": {
                    "values": [4167089, 4075182],
                    "label": "Caesarean (emergency or elective)",
                },
                "success": {
                    "values": [45883764],
                    "label": "Live birth",
                },
                "failure": {
                    "values": [37017027, 45773593, 443213],
                    "label": "Stillbirth",
                },
                "kind": "proportion",
            },
            {
                "id": "gest_length_by_delivery_mode",
                "label": "Gestational length: spontaneous vaginal vs caesarean",
                "table": "pregnancy",
                "outcome": "gestational_length_in_day",
                "cohort_column": "pregnancy_mode_delivery_concept_id",
                "reference": {
                    "values": [45885338],
                    "label": "Spontaneous vaginal",
                },
                "comparison": {
                    "values": [4167089, 4075182],
                    "label": "Caesarean (emergency or elective)",
                },
                "kind": "numeric",
            },
        ],
    },
    "tables": {
        "person": {
            "kind": "dimension",
            "view_name": "person",
            "grain": "one row per person (pregnant person or infant)",
            "pk": "person_id",
            "fks": [
                _fk("gender_concept_id", "concept.concept_id"),
                _fk("race_concept_id", "concept.concept_id"),
                _fk("ethnicity_concept_id", "concept.concept_id"),
                _fk("gender_source_concept_id", "concept.concept_id"),
                _fk("race_source_concept_id", "concept.concept_id"),
                _fk("ethnicity_source_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "person_id": _ID,
                "gender_concept_id": _CONCEPT,
                "year_of_birth": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "date_component",
                },
                "month_of_birth": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "date_component",
                },
                "day_of_birth": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "semantic_role": "date_component",
                },
                "birth_datetime": {
                    "ig_risk": "high",
                    "synthesis": "faked",
                    "semantic_role": "temporal",
                },
                "race_concept_id": _CONCEPT,
                "ethnicity_concept_id": _CONCEPT,
                "person_source_value": {
                    "ig_risk": "high",
                    "synthesis": "faked",
                    "semantic_role": "identifier",
                },
                "gender_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "gender_source_concept_id": _CONCEPT,
                "race_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "race_source_concept_id": _CONCEPT,
                "ethnicity_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "ethnicity_source_concept_id": _CONCEPT,
                "site_id": _SITE,
                "person_role": {
                    "ig_risk": "low",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
            },
        },
        "observation_period": {
            "kind": "fact",
            "view_name": "observation_period",
            "grain": "one row per observation period",
            "pk": "observation_period_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("period_type_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "observation_period_id": _ID,
                "person_id": _FK_ID,
                "observation_period_start_date": _DATE_HIGH,
                "observation_period_end_date": _DATE_HIGH,
                "period_type_concept_id": _CONCEPT,
                "site_id": _SITE,
            },
        },
        "visit_occurrence": {
            "kind": "fact",
            "view_name": "visit_occurrence",
            "grain": "one row per visit",
            "pk": "visit_occurrence_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("visit_concept_id", "concept.concept_id"),
                _fk("visit_type_concept_id", "concept.concept_id"),
                _fk("visit_source_concept_id", "concept.concept_id"),
                _fk("admitted_from_concept_id", "concept.concept_id"),
                _fk("discharged_to_concept_id", "concept.concept_id"),
                _fk(
                    "preceding_visit_occurrence_id",
                    "visit_occurrence.visit_occurrence_id",
                ),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "visit_occurrence_id": _ID,
                "person_id": _FK_ID,
                "visit_concept_id": _CONCEPT,
                "visit_start_date": _DATE_HIGH,
                "visit_start_datetime": _DATE_HIGH,
                "visit_end_date": _DATE_HIGH,
                "visit_end_datetime": _DATE_HIGH,
                "visit_type_concept_id": _CONCEPT,
                "visit_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "visit_source_concept_id": _CONCEPT,
                "admitted_from_concept_id": _CONCEPT,
                "admitted_from_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "discharged_to_concept_id": _CONCEPT,
                "discharged_to_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "preceding_visit_occurrence_id": _FK_ID,
                "site_id": _SITE,
                "pregnancy_id": _FK_ID,
            },
        },
        "visit_detail": {
            "kind": "fact",
            "view_name": "visit_detail",
            "grain": "one row per visit detail",
            "pk": "visit_detail_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("visit_detail_concept_id", "concept.concept_id"),
                _fk("visit_detail_type_concept_id", "concept.concept_id"),
                _fk("visit_detail_source_concept_id", "concept.concept_id"),
                _fk("admitted_from_concept_id", "concept.concept_id"),
                _fk("discharged_to_concept_id", "concept.concept_id"),
                _fk("preceding_visit_detail_id", "visit_detail.visit_detail_id"),
                _fk("parent_visit_detail_id", "visit_detail.visit_detail_id"),
                _fk("visit_occurrence_id", "visit_occurrence.visit_occurrence_id"),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "visit_detail_id": _ID,
                "person_id": _FK_ID,
                "visit_detail_concept_id": _CONCEPT,
                "visit_detail_start_date": _DATE_HIGH,
                "visit_detail_start_datetime": _DATE_HIGH,
                "visit_detail_end_date": _DATE_HIGH,
                "visit_detail_end_datetime": _DATE_HIGH,
                "visit_detail_type_concept_id": _CONCEPT,
                "visit_detail_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "visit_detail_source_concept_id": _CONCEPT,
                "admitted_from_concept_id": _CONCEPT,
                "admitted_from_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "discharged_to_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "discharged_to_concept_id": _CONCEPT,
                "preceding_visit_detail_id": _FK_ID,
                "parent_visit_detail_id": _FK_ID,
                "visit_occurrence_id": _FK_ID,
                "site_id": _SITE,
                "pregnancy_id": _FK_ID,
                "enrichment_tag": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "category",
                },
                "source_table": _SOURCE_TABLE,
            },
        },
        "condition_occurrence": {
            "kind": "fact",
            "view_name": "condition_occurrence",
            "grain": "one row per recorded condition occurrence",
            "pk": "condition_occurrence_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("condition_concept_id", "concept.concept_id"),
                _fk("condition_type_concept_id", "concept.concept_id"),
                _fk("condition_status_concept_id", "concept.concept_id"),
                _fk("condition_source_concept_id", "concept.concept_id"),
                _fk("visit_occurrence_id", "visit_occurrence.visit_occurrence_id"),
                _fk("visit_detail_id", "visit_detail.visit_detail_id"),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "condition_occurrence_id": _ID,
                "person_id": _FK_ID,
                "condition_concept_id": _CONCEPT,
                "condition_start_date": _DATE_HIGH,
                "condition_start_datetime": _DATE_HIGH,
                "condition_end_date": _DATE_HIGH,
                "condition_end_datetime": _DATE_HIGH,
                "condition_type_concept_id": _CONCEPT,
                "condition_status_concept_id": _CONCEPT,
                "visit_occurrence_id": _FK_ID,
                "visit_detail_id": _FK_ID,
                "condition_source_value": {
                    "ig_risk": "high",
                    "synthesis": "marginal",
                    "semantic_role": "code",
                },
                "condition_source_concept_id": _CONCEPT,
                "condition_status_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "site_id": _SITE,
                "pregnancy_id": _FK_ID,
                "source_table": _SOURCE_TABLE,
            },
        },
        "measurement": {
            "kind": "fact",
            "view_name": "measurement",
            "grain": "one row per recorded measurement",
            "pk": "measurement_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("measurement_concept_id", "concept.concept_id"),
                _fk("measurement_type_concept_id", "concept.concept_id"),
                _fk("operator_concept_id", "concept.concept_id"),
                _fk("value_as_concept_id", "concept.concept_id"),
                _fk("unit_concept_id", "concept.concept_id"),
                _fk("measurement_source_concept_id", "concept.concept_id"),
                _fk("unit_source_concept_id", "concept.concept_id"),
                _fk("meas_event_field_concept_id", "concept.concept_id"),
                _fk("visit_occurrence_id", "visit_occurrence.visit_occurrence_id"),
                _fk("visit_detail_id", "visit_detail.visit_detail_id"),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "measurement_id": _ID,
                "person_id": _FK_ID,
                "measurement_concept_id": _LOINC_CONCEPT,
                "measurement_date": _DATE_HIGH,
                "measurement_datetime": _DATE_HIGH,
                "measurement_time": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "semantic_role": "temporal",
                },
                "measurement_type_concept_id": _CONCEPT,
                "operator_concept_id": _CONCEPT,
                "value_as_number": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "unit": "unit_concept_id",
                    "semantic_role": "measure",
                },
                "value_as_concept_id": _CONCEPT,
                "unit_concept_id": _UNIT_CONCEPT,
                "range_low": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "unit": "unit_concept_id",
                    "semantic_role": "measure",
                },
                "range_high": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "unit": "unit_concept_id",
                    "semantic_role": "measure",
                },
                "visit_occurrence_id": _FK_ID,
                "visit_detail_id": _FK_ID,
                "measurement_source_value": {
                    "ig_risk": "high",
                    "synthesis": "marginal",
                    "semantic_role": "code",
                },
                "measurement_source_concept_id": _LOINC_CONCEPT,
                "unit_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "unit",
                },
                "unit_source_concept_id": _UNIT_CONCEPT,
                "value_source_value": {
                    "ig_risk": "high",
                    "synthesis": "faked",
                    "semantic_role": "free_text",
                },
                "measurement_event_id": _POLYMORPHIC_ID,
                "meas_event_field_concept_id": _CONCEPT,
                "site_id": _SITE,
                "pregnancy_id": _FK_ID,
                "enrichment_tag": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "category",
                },
                "source_table": _SOURCE_TABLE,
            },
        },
        "observation": {
            "kind": "fact",
            "view_name": "observation",
            "grain": "one row per recorded observation",
            "pk": "observation_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("observation_concept_id", "concept.concept_id"),
                _fk("observation_type_concept_id", "concept.concept_id"),
                _fk("value_as_concept_id", "concept.concept_id"),
                _fk("qualifier_concept_id", "concept.concept_id"),
                _fk("unit_concept_id", "concept.concept_id"),
                _fk("observation_source_concept_id", "concept.concept_id"),
                _fk("obs_event_field_concept_id", "concept.concept_id"),
                _fk("visit_occurrence_id", "visit_occurrence.visit_occurrence_id"),
                _fk("visit_detail_id", "visit_detail.visit_detail_id"),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "observation_id": _ID,
                "person_id": _FK_ID,
                "observation_concept_id": _CONCEPT,
                "observation_date": _DATE_HIGH,
                "observation_datetime": _DATE_HIGH,
                "observation_type_concept_id": _CONCEPT,
                "value_as_number": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "unit": "unit_concept_id",
                    "semantic_role": "measure",
                },
                "value_as_string": {
                    "ig_risk": "high",
                    "synthesis": "faked",
                    "semantic_role": "free_text",
                },
                "value_as_concept_id": _CONCEPT,
                "qualifier_concept_id": _CONCEPT,
                "unit_concept_id": _UNIT_CONCEPT,
                "visit_occurrence_id": _FK_ID,
                "visit_detail_id": _FK_ID,
                "observation_source_value": {
                    "ig_risk": "high",
                    "synthesis": "marginal",
                    "semantic_role": "code",
                },
                "observation_source_concept_id": _CONCEPT,
                "unit_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "unit",
                },
                "qualifier_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "value_source_value": {
                    "ig_risk": "high",
                    "synthesis": "faked",
                    "semantic_role": "free_text",
                },
                "observation_event_id": _POLYMORPHIC_ID,
                "obs_event_field_concept_id": _CONCEPT,
                "site_id": _SITE,
                "pregnancy_id": _FK_ID,
                "enrichment_tag": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "category",
                },
                "source_table": _SOURCE_TABLE,
            },
        },
        "procedure_occurrence": {
            "kind": "fact",
            "view_name": "procedure_occurrence",
            "grain": "one row per recorded procedure occurrence",
            "pk": "procedure_occurrence_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("procedure_concept_id", "concept.concept_id"),
                _fk("procedure_type_concept_id", "concept.concept_id"),
                _fk("modifier_concept_id", "concept.concept_id"),
                _fk("procedure_source_concept_id", "concept.concept_id"),
                _fk("visit_occurrence_id", "visit_occurrence.visit_occurrence_id"),
                _fk("visit_detail_id", "visit_detail.visit_detail_id"),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "procedure_occurrence_id": _ID,
                "person_id": _FK_ID,
                "procedure_concept_id": _CONCEPT,
                "procedure_date": _DATE_HIGH,
                "procedure_datetime": _DATE_HIGH,
                "procedure_end_date": _DATE_HIGH,
                "procedure_end_datetime": _DATE_HIGH,
                "procedure_type_concept_id": _CONCEPT,
                "modifier_concept_id": _CONCEPT,
                "quantity": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "visit_occurrence_id": _FK_ID,
                "visit_detail_id": _FK_ID,
                "procedure_source_value": {
                    "ig_risk": "high",
                    "synthesis": "marginal",
                    "semantic_role": "code",
                },
                "procedure_source_concept_id": _CONCEPT,
                "modifier_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "site_id": _SITE,
                "pregnancy_id": _FK_ID,
                "source_table": _SOURCE_TABLE,
            },
        },
        "episode": {
            "kind": "fact",
            "view_name": "episode",
            "grain": "one row per pregnancy episode",
            "pk": "episode_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("episode_concept_id", "concept.concept_id"),
                _fk("episode_object_concept_id", "concept.concept_id"),
                _fk("episode_type_concept_id", "concept.concept_id"),
                _fk("episode_source_concept_id", "concept.concept_id"),
                _fk("episode_parent_id", "episode.episode_id"),
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
            ],
            "columns": {
                "episode_id": _ID,
                "person_id": _FK_ID,
                "episode_concept_id": _CONCEPT,
                "episode_start_date": _DATE_HIGH,
                "episode_start_datetime": _DATE_HIGH,
                "episode_end_date": _DATE_HIGH,
                "episode_end_datetime": _DATE_HIGH,
                "episode_parent_id": _FK_ID,
                "episode_number": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "episode_object_concept_id": _CONCEPT,
                "episode_type_concept_id": _CONCEPT,
                "episode_source_value": {
                    "ig_risk": "high",
                    "synthesis": "marginal",
                    "semantic_role": "code",
                },
                "episode_source_concept_id": _CONCEPT,
                "pregnancy_id": _FK_ID,
                "site_id": _SITE,
            },
        },
        "episode_event": {
            "kind": "fact",
            "view_name": "episode_event",
            "grain": "one row per (episode, clinical event)",
            "pk": "episode_id",
            "pk_note": "bridge (episode_id,event_id)",
            "fks": [
                _fk("episode_id", "episode.episode_id"),
                _fk("episode_event_field_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "episode_id": _FK_ID,
                "event_id": _POLYMORPHIC_ID,
                "episode_event_field_concept_id": _CONCEPT,
                "source_table": _SOURCE_TABLE,
                "site_id": _SITE,
            },
        },
        "fact_relationship": {
            "kind": "fact",
            "view_name": "fact_relationship",
            "grain": "one row per directed fact-to-fact relationship",
            "pk": "fact_id_1",
            "pk_note": (
                "natural key (domain_concept_id_1,fact_id_1,domain_concept_id_2,"
                "fact_id_2,relationship_concept_id)"
            ),
            "fks": [],
            "columns": {
                "domain_concept_id_1": _OMOP_CONCEPT,
                "fact_id_1": _POLYMORPHIC_ID,
                "domain_concept_id_2": _OMOP_CONCEPT,
                "fact_id_2": _POLYMORPHIC_ID,
                "relationship_concept_id": _OMOP_CONCEPT,
                "site_id": _SITE,
            },
        },
        "cdm_source": {
            "kind": "reference",
            "view_name": "cdm_source",
            "grain": "one row per CDM source",
            "pk": "cdm_source_name",
            "fks": [_fk("cdm_version_concept_id", "concept.concept_id")],
            "columns": {
                "cdm_source_name": _REFERENCE_ID,
                "cdm_source_abbreviation": _REFERENCE_CATEGORY,
                "cdm_holder": _REFERENCE_LABEL,
                "source_description": _REFERENCE_LABEL,
                "source_documentation_reference": _REFERENCE_LABEL,
                "cdm_etl_reference": _REFERENCE_LABEL,
                "source_release_date": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "temporal",
                },
                "cdm_release_date": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "temporal",
                },
                "cdm_version": _REFERENCE_CATEGORY,
                "cdm_version_concept_id": _OMOP_CONCEPT,
                "vocabulary_version": _REFERENCE_CATEGORY,
            },
        },
        "pregnancy": {
            "kind": "dimension",
            "view_name": "pregnancy",
            "grain": "one row per pregnancy",
            "pk": "pregnancy_id",
            "fks": [
                _fk("person_id", "person.person_id"),
                _fk("pregnancy_outcome_concept_id", "concept.concept_id"),
                _fk("pregnancy_mode_delivery_concept_id", "concept.concept_id"),
                _fk("pregnancy_single_concept_id", "concept.concept_id"),
                _fk("pregnancy_marital_status_concept_id", "concept.concept_id"),
                _fk("prev_pregnancy_parity_concept_id", "concept.concept_id"),
                _fk("pregnancy_folic_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "person_id": _FK_ID,
                "pregnancy_id": _ID,
                "pregnancy_start_date": _DATE_HIGH,
                "pregnancy_end_date": _DATE_HIGH,
                "gestational_length_in_day": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "d",
                    "semantic_role": "measure",
                },
                "pregnancy_outcome_concept_id": _CONCEPT,
                "pregnancy_mode_delivery_concept_id": _CONCEPT,
                "pregnancy_single_concept_id": _CONCEPT,
                "pregnancy_marital_status_concept_id": _CONCEPT,
                "pregnancy_number_fetuses": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "pregnancy_number_liveborn": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "prev_pregnancy_parity_concept_id": _CONCEPT,
                "prev_pregnancy_gravidity": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "prev_livebirth_number": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "prev_still_misc_number": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "pre_pregnancy_bmi": {
                    "ig_risk": "high",
                    "synthesis": "conditional",
                    "unit": "kg/m2",
                    "semantic_role": "measure",
                },
                "pregnancy_folic_concept_id": _CONCEPT,
                "pregnancy_outcome_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "pregnancy_mode_delivery_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
                "site_id": _SITE,
                "pre_pregnancy_bmi_source": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "category",
                },
                "pregnancy_marital_status_source_value": {
                    "ig_risk": "medium",
                    "synthesis": "marginal",
                    "semantic_role": "category",
                },
            },
        },
        "infant": {
            "kind": "dimension",
            "view_name": "infant",
            "grain": "one row per infant",
            "pk": "infant_id",
            "fks": [
                _fk("pregnancy_id", "pregnancy.pregnancy_id"),
                _fk("infant_id", "person.person_id"),
                _fk("birth_outcome_concept_id", "concept.concept_id"),
                _fk("birth_con_malformation_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "pregnancy_id": _FK_ID,
                "infant_id": _ID,
                "birth_outcome_concept_id": _CONCEPT,
                "birth_weight": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "g",
                    "semantic_role": "measure",
                },
                "birth_con_malformation_concept_id": _CONCEPT,
                "birth_apgar": {
                    "ig_risk": "medium",
                    "synthesis": "conditional",
                    "unit": "{score}",
                    "semantic_role": "measure",
                },
                "labour_delivery_id": {
                    "ig_risk": "high",
                    "synthesis": "faked",
                    "semantic_role": "identifier",
                },
                "site_id": _SITE,
                "birth_weight_source": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "category",
                },
                "birth_apgar_source": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "category",
                },
            },
        },
        "concept": {
            "kind": "reference",
            "view_name": "concept",
            "grain": "one row per OMOP concept",
            "pk": "concept_id",
            "fks": [
                _fk("domain_id", "domain.domain_id"),
                _fk("vocabulary_id", "vocabulary.vocabulary_id"),
                _fk("concept_class_id", "concept_class.concept_class_id"),
            ],
            "columns": {
                "concept_id": _REFERENCE_ID,
                "concept_name": _REFERENCE_LABEL,
                "domain_id": _REFERENCE_CATEGORY,
                "vocabulary_id": _REFERENCE_CATEGORY,
                "concept_class_id": _REFERENCE_CATEGORY,
                "standard_concept": _REFERENCE_CATEGORY,
                "concept_code": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "code",
                },
                "valid_start_date": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "temporal",
                },
                "valid_end_date": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "temporal",
                },
                "invalid_reason": _REFERENCE_CATEGORY,
            },
        },
        "concept_ancestor": {
            "kind": "reference",
            "view_name": "concept_ancestor",
            "grain": "one row per (ancestor concept, descendant concept) pair",
            "pk": "ancestor_concept_id",
            "pk_note": "composite (ancestor_concept_id,descendant_concept_id)",
            "fks": [
                _fk("ancestor_concept_id", "concept.concept_id"),
                _fk("descendant_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "ancestor_concept_id": _REFERENCE_ID,
                "descendant_concept_id": _REFERENCE_ID,
                "min_levels_of_separation": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
                "max_levels_of_separation": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "unit": "{count}",
                    "semantic_role": "measure",
                },
            },
        },
        "concept_class": {
            "kind": "reference",
            "view_name": "concept_class",
            "grain": "one row per OMOP concept class",
            "pk": "concept_class_id",
            "fks": [_fk("concept_class_concept_id", "concept.concept_id")],
            "columns": {
                "concept_class_id": _REFERENCE_ID,
                "concept_class_name": _REFERENCE_LABEL,
                "concept_class_concept_id": _REFERENCE_ID,
            },
        },
        "concept_relationship": {
            "kind": "reference",
            "view_name": "concept_relationship",
            "grain": "one row per (concept 1, concept 2, relationship) tuple",
            "pk": "concept_id_1",
            "pk_note": "composite (concept_id_1,concept_id_2,relationship_id)",
            "fks": [
                _fk("concept_id_1", "concept.concept_id"),
                _fk("concept_id_2", "concept.concept_id"),
                _fk("relationship_id", "relationship.relationship_id"),
            ],
            "columns": {
                "concept_id_1": _REFERENCE_ID,
                "concept_id_2": _REFERENCE_ID,
                "relationship_id": _REFERENCE_CATEGORY,
                "valid_start_date": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "temporal",
                },
                "valid_end_date": {
                    "ig_risk": "low",
                    "synthesis": "constant",
                    "semantic_role": "temporal",
                },
                "invalid_reason": _REFERENCE_CATEGORY,
            },
        },
        "concept_synonym": {
            "kind": "reference",
            "view_name": "concept_synonym",
            "grain": "one row per concept synonym and language",
            "pk": "concept_id",
            "pk_note": "composite (concept_id,concept_synonym_name,language_concept_id)",
            "fks": [
                _fk("concept_id", "concept.concept_id"),
                _fk("language_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "concept_id": _REFERENCE_ID,
                "concept_synonym_name": _REFERENCE_LABEL,
                "language_concept_id": _REFERENCE_ID,
            },
        },
        "domain": {
            "kind": "reference",
            "view_name": "domain",
            "grain": "one row per OMOP domain",
            "pk": "domain_id",
            "fks": [_fk("domain_concept_id", "concept.concept_id")],
            "columns": {
                "domain_id": _REFERENCE_ID,
                "domain_name": _REFERENCE_LABEL,
                "domain_concept_id": _REFERENCE_ID,
            },
        },
        "vocabulary": {
            "kind": "reference",
            "view_name": "vocabulary",
            "grain": "one row per OMOP vocabulary",
            "pk": "vocabulary_id",
            "fks": [_fk("vocabulary_concept_id", "concept.concept_id")],
            "columns": {
                "vocabulary_id": _REFERENCE_ID,
                "vocabulary_name": _REFERENCE_LABEL,
                "vocabulary_reference": _REFERENCE_LABEL,
                "vocabulary_version": _REFERENCE_CATEGORY,
                "vocabulary_concept_id": _REFERENCE_ID,
            },
        },
        "relationship": {
            "kind": "reference",
            "view_name": "relationship",
            "grain": "one row per OMOP relationship type",
            "pk": "relationship_id",
            "fks": [
                _fk("reverse_relationship_id", "relationship.relationship_id"),
                _fk("relationship_concept_id", "concept.concept_id"),
            ],
            "columns": {
                "relationship_id": _REFERENCE_ID,
                "relationship_name": _REFERENCE_LABEL,
                "is_hierarchical": _REFERENCE_CATEGORY,
                "defines_ancestry": _REFERENCE_CATEGORY,
                "reverse_relationship_id": _REFERENCE_ID,
                "relationship_concept_id": _REFERENCE_ID,
            },
        },
    },
    "relationships": [
        {
            "name": "pregnancy_person",
            "from": "pregnancy.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "infant_pregnancy",
            "from": "infant.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "infant_person",
            "from": "infant.infant_id",
            "to": "person.person_id",
            "cardinality": "one_to_one",
        },
        {
            "name": "observation_period_person",
            "from": "observation_period.person_id",
            "to": "person.person_id",
            "cardinality": "one_to_one",
        },
        {
            "name": "visit_person",
            "from": "visit_occurrence.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_pregnancy",
            "from": "visit_occurrence.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_preceding",
            "from": "visit_occurrence.preceding_visit_occurrence_id",
            "to": "visit_occurrence.visit_occurrence_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_detail_person",
            "from": "visit_detail.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_detail_visit",
            "from": "visit_detail.visit_occurrence_id",
            "to": "visit_occurrence.visit_occurrence_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_detail_pregnancy",
            "from": "visit_detail.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_detail_preceding",
            "from": "visit_detail.preceding_visit_detail_id",
            "to": "visit_detail.visit_detail_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "visit_detail_parent",
            "from": "visit_detail.parent_visit_detail_id",
            "to": "visit_detail.visit_detail_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "condition_person",
            "from": "condition_occurrence.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "condition_visit",
            "from": "condition_occurrence.visit_occurrence_id",
            "to": "visit_occurrence.visit_occurrence_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "condition_pregnancy",
            "from": "condition_occurrence.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "measurement_person",
            "from": "measurement.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "measurement_visit",
            "from": "measurement.visit_occurrence_id",
            "to": "visit_occurrence.visit_occurrence_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "measurement_pregnancy",
            "from": "measurement.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "observation_person",
            "from": "observation.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "observation_visit",
            "from": "observation.visit_occurrence_id",
            "to": "visit_occurrence.visit_occurrence_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "observation_pregnancy",
            "from": "observation.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "procedure_person",
            "from": "procedure_occurrence.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "procedure_visit",
            "from": "procedure_occurrence.visit_occurrence_id",
            "to": "visit_occurrence.visit_occurrence_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "procedure_pregnancy",
            "from": "procedure_occurrence.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "episode_person",
            "from": "episode.person_id",
            "to": "person.person_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "episode_pregnancy",
            "from": "episode.pregnancy_id",
            "to": "pregnancy.pregnancy_id",
            "cardinality": "one_to_one",
        },
        {
            "name": "episode_parent",
            "from": "episode.episode_parent_id",
            "to": "episode.episode_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "episode_event_episode",
            "from": "episode_event.episode_id",
            "to": "episode.episode_id",
            "cardinality": "many_to_one",
        },
        {
            "name": "episode_event_event",
            "from": "episode_event.event_id",
            "polymorphic": True,
            "discriminator": "source_table",
            "targets": [
                "measurement.measurement_id",
                "observation.observation_id",
                "condition_occurrence.condition_occurrence_id",
                "procedure_occurrence.procedure_occurrence_id",
                "visit_occurrence.visit_occurrence_id",
            ],
            "cardinality": "many_to_one",
        },
        {
            "name": "fact_rel",
            "from": "fact_relationship.fact_id_1",
            "polymorphic": True,
            "discriminator": "domain_concept_id_1",
            "targets": [
                "person.person_id",
                "observation_period.observation_period_id",
                "visit_occurrence.visit_occurrence_id",
                "visit_detail.visit_detail_id",
                "condition_occurrence.condition_occurrence_id",
                "measurement.measurement_id",
                "observation.observation_id",
                "procedure_occurrence.procedure_occurrence_id",
                "episode.episode_id",
                "pregnancy.pregnancy_id",
                "infant.infant_id",
            ],
            "cardinality": "many_to_many",
        },
        {
            "name": "fact_rel_target",
            "from": "fact_relationship.fact_id_2",
            "polymorphic": True,
            "discriminator": "domain_concept_id_2",
            "targets": [
                "person.person_id",
                "observation_period.observation_period_id",
                "visit_occurrence.visit_occurrence_id",
                "visit_detail.visit_detail_id",
                "condition_occurrence.condition_occurrence_id",
                "measurement.measurement_id",
                "observation.observation_id",
                "procedure_occurrence.procedure_occurrence_id",
                "episode.episode_id",
                "pregnancy.pregnancy_id",
                "infant.infant_id",
            ],
            "cardinality": "many_to_many",
        },
    ],
}

In [0]:

def _qi(value):
    return "`" + value.replace("`", "``") + "`"


def _ql(value):
    return "'" + value.replace("'", "''") + "'"


def _qcol(value):
    return "`" + value.replace("`", "``") + "`"


def _fqn(catalog, schema, table):
    return f"{_qi(catalog)}.{_qi(schema)}.{_qi(table)}"


config_errors = validate_config(DATASET_CONFIG)
assert not config_errors, "config invalid:\n" + "\n".join(config_errors)
assert DATASET_CONFIG["dataset_name"] == P["dataset_name"]
logical = list(DATASET_CONFIG["tables"])
assert len(logical) == 22, f"expected 22 logical tables, got {len(logical)}"
assert not any(name.startswith("_") or name.startswith(P["synthetic_prefix"]) for name in logical)
_mark("config_validated", tables=len(logical))


def _table_inventory(catalog, schema):
    return {
        row.table_name: row
        for row in spark.sql(
            f"""SELECT table_name, table_type, comment
                FROM {_qi(catalog)}.information_schema.tables
                WHERE table_schema={_ql(schema)}"""
        ).collect()
    }


real_inventory = _table_inventory(rcat, rsch)
synth_inventory = _table_inventory(scat, ssch)
expected_synth = {f"{P['synthetic_prefix']}{table}" for table in logical}
for table in logical:
    assert table in real_inventory, f"missing real table {table}"
    assert f"{P['synthetic_prefix']}{table}" in synth_inventory, f"missing synth table {table}"
available_synth = {
    name for name in synth_inventory if name.startswith(P["synthetic_prefix"])
}
assert available_synth == expected_synth, (
    f"unexpected synthetic inventory: missing={sorted(expected_synth - available_synth)} "
    f"extra={sorted(available_synth - expected_synth)}"
)
_mark("inventory_validated", real=len(real_inventory), synthetic=len(synth_inventory))


def _columns(catalog, schema):
    by_table = {}
    rows = spark.sql(
        f"""SELECT table_name, column_name, ordinal_position, full_data_type AS dtype,
                   is_nullable, comment
            FROM {_qi(catalog)}.information_schema.columns
            WHERE table_schema={_ql(schema)}
            ORDER BY table_name, ordinal_position"""
    ).collect()
    for row in rows:
        by_table.setdefault(row.table_name, []).append(row)
    return by_table


real_columns = _columns(rcat, rsch)
synth_columns = _columns(scat, ssch)
for table in logical:
    real_names = [row.column_name for row in real_columns[table]]
    synth_name = f"{P['synthetic_prefix']}{table}"
    synth_names = [row.column_name for row in synth_columns[synth_name]]
    assert real_names == synth_names, f"column name/order mismatch for {table}"
_mark("columns_validated")


def _latest_history(catalog, schema, table):
    row = spark.sql(f"DESCRIBE HISTORY {_fqn(catalog, schema, table)} LIMIT 1").first()
    return {"version": int(row["version"]), "timestamp": str(row["timestamp"])}


def _detail(catalog, schema, table):
    row = spark.sql(f"DESCRIBE DETAIL {_fqn(catalog, schema, table)}").first().asDict()
    return {
        "delta_id": row.get("id"),
        "format": row.get("format"),
        "size_bytes": int(row.get("sizeInBytes") or 0),
        "num_files": int(row.get("numFiles") or 0),
    }


def _pinned(catalog, schema, table, version):
    return spark.sql(
        f"SELECT * FROM {_fqn(catalog, schema, table)} VERSION AS OF {int(version)}"
    )


snapshot = {}
real_dfs, synth_dfs = {}, {}
for table in logical:
    _mark("snapshot_table_start", table=table)
    synth_table = f"{P['synthetic_prefix']}{table}"
    real_history = _latest_history(rcat, rsch, table)
    synth_history = _latest_history(scat, ssch, synth_table)
    real_detail = _detail(rcat, rsch, table)
    synth_detail = _detail(scat, ssch, synth_table)
    real_df = _pinned(rcat, rsch, table, real_history["version"])
    synth_df = _pinned(scat, ssch, synth_table, synth_history["version"])
    real_rows = real_df.count()
    real_dfs[table], synth_dfs[table] = real_df, synth_df
    snapshot[table] = {
        "real": {
            "uc_table": f"{rcat}.{rsch}.{table}",
            "delta_version": real_history["version"],
            "as_of": real_history["timestamp"],
            "rows": real_rows,
            **real_detail,
        },
        "synth": {
            "uc_table": f"{scat}.{ssch}.{synth_table}",
            "delta_version": synth_history["version"],
            "as_of": synth_history["timestamp"],
            "rows": None,
            **synth_detail,
        },
    }
    _mark("snapshot_table_done", table=table, real_rows=real_rows)

untagged_all = []
for table in logical:
    _mark("normalize_table_start", table=table)
    configured = DATASET_CONFIG["tables"][table].get("columns", {})
    for row in real_columns[table]:
        if "ig_risk" not in configured.get(row.column_name, {}):
            untagged_all.append(f"{table}.{row.column_name}")
if P["stats_basis"] == "real_suppressed":
    assert not untagged_all, (
        "real_suppressed is governance-gated until every column has an explicit "
        f"ig_risk classification; {len(untagged_all)} remain"
    )

In [0]:
_SPARK_STORAGE_TYPE = {
    "int64": "bigint",
    "float64": "double",
    "string": "string",
    "bool": "boolean",
    # Serverless fixes TIMESTAMP_NTZ to Parquet INT64 microseconds; regular
    # TIMESTAMP is written as INT96 and fails the package contract.
    "timestamp": "timestamp_ntz",
    "date": "date",
}
normalized = {}
table_plans = {}
effective_table_configs = {}
tables_block = {}
excluded_all = []
coercions = []

for table in logical:
    synth_table = f"{P['synthetic_prefix']}{table}"
    source_df = synth_dfs[table]
    synth_by_name = {row.column_name: row for row in synth_columns[synth_table]}
    select_exprs, check_exprs, check_specs, plans, cols_meta = [], [], [], [], []
    for index, real_column in enumerate(real_columns[table]):
        synth_column = synth_by_name[real_column.column_name]
        plan = plan_cast(real_column.column_name, real_column.dtype, synth_column.dtype)
        plans.append(plan)
        cols_meta.append(
            {
                "name": real_column.column_name,
                "ordinal": int(real_column.ordinal_position),
                "real_type": real_column.dtype,
                "synth_type": synth_column.dtype,
                "nullable": str(real_column.is_nullable).upper() == "YES",
                "comment": real_column.comment,
            }
        )
        if plan["action"] == "exclude":
            continue
        raw = F.col(_qcol(real_column.column_name))
        logical_value = raw
        if plan["action"] == "cast":
            logical_value = F.expr(
                f"try_cast({_qcol(real_column.column_name)} AS {plan['cast_to']})"
            )
            coercions.append(
                {
                    "table": table,
                    "column": real_column.column_name,
                    "from": synth_column.dtype,
                    "to": real_column.dtype,
                    "published_type": plan["published_type"],
                    "null_introduction_failures": 0,
                }
            )
        if plan["check_midnight_only"]:
            alias = f"midnight_{index}"
            check_exprs.append(
                F.sum(
                    F.when(
                        raw.isNotNull() & (raw != F.date_trunc("day", raw)), 1
                    ).otherwise(0)
                ).alias(alias)
            )
            check_specs.append((alias, real_column.column_name, "non-midnight timestamp"))
        if plan["check_null_introduction"]:
            alias = f"nullcast_{index}"
            check_exprs.append(
                F.sum(
                    F.when(raw.isNotNull() & logical_value.isNull(), 1).otherwise(0)
                ).alias(alias)
            )
            check_specs.append((alias, real_column.column_name, "introduced null"))
        select_exprs.append(
            logical_value.cast(_SPARK_STORAGE_TYPE[plan["published_type"]]).alias(
                real_column.column_name
            )
        )
    if check_exprs:
        check_row = source_df.agg(*check_exprs).first().asDict()
        for alias, column_name, failure_kind in check_specs:
            failures = int(check_row.get(alias) or 0)
            assert failures == 0, f"{table}.{column_name}: {failures} {failure_kind} rows"
    out_df = source_df.select(*select_exprs)
    synth_rows = out_df.count()
    snapshot[table]["synth"]["rows"] = synth_rows
    target_bytes = P["target_file_mb"] * 1024 * 1024
    partitions = max(1, math.ceil(snapshot[table]["synth"]["size_bytes"] / target_bytes))
    (
        out_df.coalesce(partitions)
        .write.mode("overwrite")
        .option("compression", "snappy")
        .option("outputTimestampType", "TIMESTAMP_MICROS")
        .parquet(f"{staging}/parquet/{table}")
    )
    normalized[table] = out_df
    table_plans[table] = plans

    table_config = deepcopy(DATASET_CONFIG["tables"][table])
    table_config["description"] = real_inventory[table].comment or ""
    table_config.setdefault("columns", {})
    for real_column in real_columns[table]:
        table_config["columns"].setdefault(real_column.column_name, {}).setdefault(
            "synthesis", "upstream_prebuilt"
        )
    effective_table_configs[table] = table_config
    table_entry, excluded = build_table_entry(
        table,
        table_config,
        cols_meta,
        synth_rows=synth_rows,
        source_rows=snapshot[table]["real"]["rows"],
    )
    tables_block[table] = table_entry
    excluded_all.extend(excluded)
    _mark("normalize_table_done", table=table, synth_rows=synth_rows)

assert len(coercions) == 31, f"PREBORN expected 31 physical coercions, got {len(coercions)}"

In [0]:

def _base_profile(df, column_names):
    expressions = []
    for index, column_name in enumerate(column_names):
        column = F.col(_qcol(column_name))
        expressions.extend(
            [
                F.sum(F.when(column.isNull(), 1).otherwise(0)).alias(f"n_{index}"),
                F.approx_count_distinct(column).alias(f"d_{index}"),
            ]
        )
    row = df.agg(*expressions).first().asDict()
    return {
        column_name: {
            "null_count": int(row.get(f"n_{index}") or 0),
            "distinct_count": int(row.get(f"d_{index}") or 0),
        }
        for index, column_name in enumerate(column_names)
    }


def _is_concept_column(column_name):
    return (
        column_name == "concept_id"
        or column_name.endswith("_concept_id")
        or column_name in {"concept_id_1", "concept_id_2", "ancestor_concept_id", "descendant_concept_id"}
    )


def _value_counts(df, column_name):
    return {
        row["value"]: int(row["count"])
        for row in (
            df.where(F.col(_qcol(column_name)).isNotNull())
            .groupBy(F.col(_qcol(column_name)).alias("value"))
            .count()
            .collect()
        )
    }


def _length_profile(df, column_name):
    lengths = (
        df.select(F.length(F.col(_qcol(column_name))).cast("double").alias("length"))
        .where(F.col("length").isNotNull())
    )
    row = lengths.agg(F.min("length").alias("lo"), F.max("length").alias("hi")).first()
    median = lengths.approxQuantile("length", [0.5], 0.01)
    return {
        "length_min": int(row.lo) if row.lo is not None else None,
        "length_median": int(round(median[0])) if median else None,
        "length_max": int(row.hi) if row.hi is not None else None,
    }


def _numeric_histogram(df, column_name, low, high, bins=10):
    if low is None or high is None:
        return []
    low, high = float(low), float(high)
    if not math.isfinite(low) or not math.isfinite(high):
        return []
    column = F.col(_qcol(column_name)).cast("double")
    if high <= low:
        count = df.where(column.isNotNull()).count()
        return [(f"{low:.6g}", count)] if count else []
    width = (high - low) / bins
    bucket = F.when(column <= low, F.lit(0)).when(column >= high, F.lit(bins - 1)).otherwise(
        F.floor((column - F.lit(low)) / F.lit(width)).cast("int")
    )
    counts = {
        int(row.bucket): int(row["count"])
        for row in (
            df.where(column.isNotNull())
            .select(bucket.alias("bucket"))
            .groupBy("bucket")
            .count()
            .collect()
        )
    }
    return [
        (f"{low + i * width:.6g}-{low + (i + 1) * width:.6g}", counts.get(i, 0))
        for i in range(bins)
    ]


def compute_column_agg(
    df, table_name, column_meta, column_schema, column_config, base, row_count, basis
):
    column_name = column_meta["name"]
    published_type = column_meta["type"]
    agg = {"row_count": row_count, **base[column_name]}
    risk = column_meta["ig_risk"]
    if basis == "real" and risk == "high":
        return agg

    role = column_config.get("semantic_role")
    concept_column = _is_concept_column(column_name)
    reference_category = column_name in {
        "domain_id",
        "vocabulary_id",
        "concept_class_id",
        "relationship_id",
        "site_id",
        "source_table",
        "person_role",
    }
    identifier = bool(column_meta.get("pk") or column_meta.get("fk")) or role == "identifier"
    category = concept_column or reference_category or role == "category" or bool(
        column_config.get("code_system")
    )
    free_text = role == "free_text"

    if published_type in ("date", "timestamp"):
        month = F.date_format(F.date_trunc("month", F.col(_qcol(column_name))), "yyyy-MM")
        row = df.agg(F.min(month).alias("lo"), F.max(month).alias("hi")).first()
        agg["min_month"], agg["max_month"] = row.lo, row.hi
        agg["month_histogram"] = [
            (row.month, int(row["count"]))
            for row in (
                df.where(F.col(_qcol(column_name)).isNotNull())
                .select(month.alias("month"))
                .groupBy("month")
                .count()
                .orderBy("month")
                .collect()
            )
        ]
        return agg

    if published_type == "bool":
        row = df.agg(
            F.sum(F.when(F.col(_qcol(column_name)) == F.lit(True), 1).otherwise(0)).alias("t"),
            F.sum(F.when(F.col(_qcol(column_name)) == F.lit(False), 1).otherwise(0)).alias("f"),
        ).first()
        agg["true_count"], agg["false_count"] = int(row.t or 0), int(row.f or 0)
        return agg

    if category and agg["distinct_count"] <= 5000:
        agg["value_counts"] = _value_counts(df, column_name)
        return agg

    if identifier or free_text or (
        published_type == "string" and agg["distinct_count"] > 200
    ):
        agg["high_cardinality"] = True
        if published_type == "string":
            agg.update(_length_profile(df, column_name))
        return agg

    if published_type in ("int64", "float64"):
        column = F.col(_qcol(column_name)).cast("double")
        row = df.agg(F.avg(column).alias("mean"), F.stddev_samp(column).alias("std")).first()
        quantiles = df.approxQuantile(column_name, [0.05, 0.25, 0.5, 0.75, 0.95], 0.01)
        agg.update(
            {
                "mean": float(row.mean) if row.mean is not None else None,
                "std": float(row.std) if row.std is not None else None,
            }
        )
        for key, value in zip(("p5", "p25", "p50", "p75", "p95"), quantiles):
            agg[key] = float(value)
        if quantiles:
            agg["histogram"] = _numeric_histogram(
                df, column_name, quantiles[0], quantiles[-1]
            )
        return agg

    if published_type == "string":
        agg["value_counts"] = _value_counts(df, column_name)
        return agg

    agg["high_cardinality"] = True
    return agg


basis = "synthetic" if P["stats_basis"] == "synthetic" else "real"
stats_sources = normalized if basis == "synthetic" else real_dfs
label_sources = stats_sources
concept_labels = {
    row.concept_id: row.concept_name
    for row in label_sources["concept"].select("concept_id", "concept_name").collect()
}
domain_labels = {
    row.domain_id: row.domain_name
    for row in label_sources["domain"].select("domain_id", "domain_name").collect()
}
vocabulary_labels = {
    row.vocabulary_id: row.vocabulary_name
    for row in label_sources["vocabulary"].select("vocabulary_id", "vocabulary_name").collect()
}
class_labels = {
    row.concept_class_id: row.concept_class_name
    for row in label_sources["concept_class"].select(
        "concept_class_id", "concept_class_name"
    ).collect()
}
relationship_labels = {
    row.relationship_id: row.relationship_name
    for row in label_sources["relationship"].select(
        "relationship_id", "relationship_name"
    ).collect()
}


def _labels_for(column_name):
    if _is_concept_column(column_name):
        return concept_labels
    return {
        "domain_id": domain_labels,
        "vocabulary_id": vocabulary_labels,
        "concept_class_id": class_labels,
        "relationship_id": relationship_labels,
    }.get(column_name, {})


def _compute_table_stats(table):
    df = stats_sources[table]
    published_columns = [
        column for column in tables_block[table]["columns"] if column["type"] != "excluded"
    ]
    names = [column["name"] for column in published_columns]
    base_profile = _base_profile(df, names)
    config_columns = effective_table_configs[table].get("columns", {})
    shaped = {}
    for column in published_columns:
        column_name = column["name"]
        agg = compute_column_agg(
            df,
            table,
            column,
            column,
            config_columns.get(column_name, {}),
            base_profile,
            snapshot[table]["synth" if basis == "synthetic" else "real"]["rows"],
            basis,
        )
        shaped[column_name] = shape_column_stats(
            basis=basis,
            ig_risk=column["ig_risk"],
            published_type=column["type"],
            agg=agg,
            k=P["k"],
            labels=_labels_for(column_name),
        )
    document = {
        "table": table,
        "basis": basis,
        "source_row_count_rounded": round_to_5(
            snapshot[table]["synth" if basis == "synthetic" else "real"]["rows"]
        ),
        "computed_at": run_at,
        "suppression_threshold": P["k"],
        "columns": shaped,
    }
    dbutils.fs.put(
        f"{staging}/stats/{table}.json",
        json.dumps(document, indent=2, default=str),
        True,
    )
    return document


stats_docs = {}
for table in logical:
    _mark("stats_table_start", table=table)
    try:
        stats_docs[table] = _compute_table_stats(table)
    except Exception:
        _record_failure("stats_table_error", table=table)
        raise
    _mark("stats_table_done", table=table)


def _cohort_arm_expression(recipe):
    cohort = F.col(_qcol(recipe["cohort_column"]))
    return (
        F.when(
            cohort.isin(*recipe["reference"]["values"]),
            F.lit("reference"),
        )
        .when(
            cohort.isin(*recipe["comparison"]["values"]),
            F.lit("comparison"),
        )
        .otherwise(F.lit(None).cast("string"))
    )


def _comparison_base(recipe):
    return {
        "id": recipe["id"],
        "label": recipe["label"],
        "kind": recipe["kind"],
        "table": recipe["table"],
        "outcome": recipe["outcome"],
    }


def _public_cohort(label, stats):
    cohort = {"label": label}
    if stats["n"] >= P["k"]:
        cohort["n"] = round_to_5(stats["n"])
    else:
        cohort["count_suppressed"] = True
    return cohort


def _insufficient_comparison(recipe, stats, minimum_n):
    result = _comparison_base(recipe)
    result.update(
        {
            "reference": _public_cohort(
                recipe["reference"]["label"], stats["reference"]
            ),
            "comparison": _public_cohort(
                recipe["comparison"]["label"], stats["comparison"]
            ),
            "sufficient": False,
            "status": "insufficient",
            "reason": (
                f"Each cohort requires at least {minimum_n} usable rows "
                "after suppression."
            ),
        }
    )
    return result


def _validate_runtime_insight_recipe(recipe):
    table_columns = {
        column["name"]: column
        for column in tables_block[recipe["table"]]["columns"]
    }
    outcome = table_columns[recipe["outcome"]]
    cohort = table_columns[recipe["cohort_column"]]
    assert outcome["ig_risk"] != "high", (
        f"{recipe['id']}: high-risk outcome is forbidden"
    )
    assert cohort["ig_risk"] != "high", (
        f"{recipe['id']}: high-risk cohort is forbidden"
    )
    if recipe["kind"] == "numeric":
        assert outcome["type"] in ("int64", "float64"), (
            f"{recipe['id']}: numeric outcome has type {outcome['type']}"
        )
    else:
        assert outcome["type"] in ("int64", "string", "bool"), (
            f"{recipe['id']}: proportion outcome has type {outcome['type']}"
        )


def _compute_numeric_insight(recipe):
    value = F.col(_qcol(recipe["outcome"])).cast("double")
    profiled = (
        real_dfs[recipe["table"]]
        .select(
            _cohort_arm_expression(recipe).alias("arm"),
            value.alias("value"),
        )
        .where(
            F.col("arm").isNotNull()
            & F.col("value").isNotNull()
            & ~F.isnan(F.col("value"))
        )
        .groupBy("arm")
        .agg(
            F.count("value").alias("n"),
            F.avg("value").alias("mean"),
            F.stddev_samp("value").alias("sd"),
        )
        .collect()
    )
    stats = {
        "reference": {"n": 0, "mean": None, "sd": None},
        "comparison": {"n": 0, "mean": None, "sd": None},
    }
    for row in profiled:
        stats[row["arm"]] = {
            "n": int(row["n"]),
            "mean": float(row["mean"]) if row["mean"] is not None else None,
            "sd": float(row["sd"]) if row["sd"] is not None else None,
        }

    minimum_n = max(INSIGHT_MIN_COHORT_N, P["k"])
    if any(stats[arm]["n"] < minimum_n for arm in stats):
        return _insufficient_comparison(recipe, stats, minimum_n)

    difference = welch_numeric_difference(
        stats["reference"], stats["comparison"]
    )
    result = _comparison_base(recipe)
    result.update(
        {
            "reference": {
                "label": recipe["reference"]["label"],
                "n": round_to_5(stats["reference"]["n"]),
                "mean": _public_metric(stats["reference"]["mean"], 2),
                "sd": _public_metric(stats["reference"]["sd"], 2),
            },
            "comparison": {
                "label": recipe["comparison"]["label"],
                "n": round_to_5(stats["comparison"]["n"]),
                "mean": _public_metric(stats["comparison"]["mean"], 2),
                "sd": _public_metric(stats["comparison"]["sd"], 2),
            },
            "difference": {
                "estimate": _public_metric(difference["estimate"], 2),
                "ci95": [
                    _public_metric(difference["ci95"][0], 2),
                    _public_metric(difference["ci95"][1], 2),
                ],
                "hedges_g": _public_metric(difference["hedges_g"], 3),
            },
            "sufficient": True,
        }
    )
    return result


def _compute_proportion_insight(recipe):
    outcome = F.col(_qcol(recipe["outcome"]))
    binary_value = (
        F.when(outcome.isin(*recipe["success"]["values"]), F.lit(1))
        .when(outcome.isin(*recipe["failure"]["values"]), F.lit(0))
        .otherwise(F.lit(None).cast("int"))
    )
    profiled = (
        real_dfs[recipe["table"]]
        .select(
            _cohort_arm_expression(recipe).alias("arm"),
            binary_value.alias("value"),
        )
        .where(F.col("arm").isNotNull() & F.col("value").isNotNull())
        .groupBy("arm")
        .agg(
            F.count("value").alias("n"),
            F.sum("value").alias("successes"),
        )
        .collect()
    )
    stats = {
        "reference": {"n": 0, "successes": 0},
        "comparison": {"n": 0, "successes": 0},
    }
    for row in profiled:
        stats[row["arm"]] = {
            "n": int(row["n"]),
            "successes": int(row["successes"] or 0),
        }

    minimum_n = max(INSIGHT_MIN_COHORT_N, P["k"])
    if any(stats[arm]["n"] < minimum_n for arm in stats):
        return _insufficient_comparison(recipe, stats, minimum_n)

    difference = newcombe_proportion_difference(
        stats["reference"], stats["comparison"]
    )
    result = _comparison_base(recipe)
    result.update(
        {
            "reference": {
                "label": recipe["reference"]["label"],
                "n": round_to_5(stats["reference"]["n"]),
                "proportion": _public_metric(
                    stats["reference"]["successes"] / stats["reference"]["n"],
                    4,
                ),
            },
            "comparison": {
                "label": recipe["comparison"]["label"],
                "n": round_to_5(stats["comparison"]["n"]),
                "proportion": _public_metric(
                    stats["comparison"]["successes"] / stats["comparison"]["n"],
                    4,
                ),
            },
            "difference": {
                "estimate": _public_metric(difference["estimate"], 4),
                "ci95": [
                    _public_metric(difference["ci95"][0], 4),
                    _public_metric(difference["ci95"][1], 4),
                ],
            },
            "sufficient": True,
        }
    )
    return result


def _compute_insight_comparison(recipe):
    _validate_runtime_insight_recipe(recipe)
    if recipe["kind"] == "numeric":
        return _compute_numeric_insight(recipe)
    if recipe["kind"] == "proportion":
        return _compute_proportion_insight(recipe)
    raise AssertionError(f"unsupported insight kind {recipe['kind']}")


comparison_recipes = deepcopy(
    DATASET_CONFIG.get("insights", {}).get("comparisons", [])
)
comparison_results = []
comparisons_doc = None
if comparison_recipes:
    for recipe in comparison_recipes:
        _mark("comparison_start", comparison_id=recipe["id"])
        try:
            comparison_results.append(_compute_insight_comparison(recipe))
        except Exception:
            _record_failure("comparison_error", comparison_id=recipe["id"])
            raise
        _mark("comparison_done", comparison_id=recipe["id"])

    comparisons_doc = {
        "basis": "real_suppressed",
        "computed_at": run_at,
        "results": comparison_results,
    }
    dbutils.fs.put(
        f"{staging}/comparisons.json",
        json.dumps(comparisons_doc, indent=2, default=str),
        True,
    )


In [0]:
dataset_block = build_dataset_block(
    P["dataset_name"],
    DATASET_CONFIG["description"],
    run_at[:10],
    run_at,
    P["seed"],
    basis,
    owner=DATASET_CONFIG.get("owner"),
    source_system=DATASET_CONFIG.get("source_system"),
    refresh_cadence=DATASET_CONFIG.get("refresh_cadence"),
    insights=DATASET_CONFIG.get("insights"),
)
schema_doc = {
    "dataset": dataset_block,
    "stats": {"basis": basis},
    "tables": tables_block,
    "relationships": DATASET_CONFIG.get("relationships", []),
}
schema_text = yaml.safe_dump(schema_doc, sort_keys=False, allow_unicode=True)
dbutils.fs.put(f"{staging}/schema.yaml", schema_text, True)

In [0]:
validation = {"status": "running", "gates": {}}
_mark("validation_start")


def gate(name, passed, detail=""):
    validation["gates"][name] = {
        "status": "pass" if passed else "fail",
        "detail": str(detail),
    }
    if not passed:
        _record_failure("validation_gate_error", gate=name, detail=detail)
    assert passed, f"validation gate failed: {name}: {detail}"


parsed_schema = yaml.safe_load(dbutils.fs.head(f"{staging}/schema.yaml", 10_000_000))
gate("schema_yaml_parses", parsed_schema["dataset"]["name"] == P["dataset_name"])
gate("table_inventory", set(parsed_schema["tables"]) == set(logical), len(logical))
gate("stats_basis", parsed_schema["stats"]["basis"] == basis, basis)
if comparison_recipes:
    parsed_insights = parsed_schema["dataset"].get("insights", {})
    gate(
        "insights_default_lens",
        parsed_insights.get("default_lens")
        == DATASET_CONFIG["insights"]["default_lens"],
    )
    gate(
        "insights_recipe_ids",
        [recipe["id"] for recipe in parsed_insights.get("comparisons", [])]
        == [recipe["id"] for recipe in comparison_recipes],
    )
    reread_comparisons = json.loads(
        dbutils.fs.head(f"{staging}/comparisons.json", 10_000_000)
    )
    comparison_errors = comparison_document_errors(
        reread_comparisons, comparison_recipes
    )
    gate(
        "comparisons_json",
        not comparison_errors,
        comparison_errors,
    )

staged = {}
total_columns = 0
try:
    import pyarrow.parquet as pq
except ModuleNotFoundError:
    pq = None


def _first_parquet_file(path):
    for item in dbutils.fs.ls(path):
        if item.isDir():
            found = _first_parquet_file(item.path)
            if found:
                return found
        elif item.name.endswith(".parquet"):
            return item.path.removeprefix("dbfs:")
    return None


def _is_driver_local_path(path):
    return path.startswith(("/Volumes/", "/dbfs/", "/tmp/", "file:/"))


for table in logical:
    df = spark.read.parquet(f"{staging}/parquet/{table}")
    staged[table] = df
    expected = [
        column for column in tables_block[table]["columns"] if column["type"] != "excluded"
    ]
    total_columns += len(tables_block[table]["columns"])
    gate(
        f"schema_names_{table}",
        df.columns == [column["name"] for column in expected],
        df.columns,
    )
    actual_types = [map_storage_type(field.dataType.simpleString()) for field in df.schema.fields]
    gate(
        f"schema_types_{table}",
        actual_types == [column["type"] for column in expected],
        actual_types,
    )
    gate(
        f"row_count_{table}",
        df.count() == tables_block[table]["rows"],
        tables_block[table]["rows"],
    )
    gate(
        f"atomic_types_{table}",
        all(map_storage_type(field.dataType.simpleString()) != "excluded" for field in df.schema.fields),
    )
    parquet_file = _first_parquet_file(f"{staging}/parquet/{table}")
    gate(f"parquet_file_present_{table}", parquet_file is not None)
    if _is_driver_local_path(parquet_file):
        gate("pyarrow_available_for_footer_audit", pq is not None)
        physical_schema = str(pq.ParquetFile(parquet_file).schema).lower()
        gate(f"no_int96_{table}", "int96" not in physical_schema, physical_schema)
    else:
        # PyArrow does not inherit Unity Catalog's scoped ABFSS credential. Giving
        # it an abfss:// URI triggers Azure's default credential chain (including
        # the unavailable Azure CLI on serverless). Production therefore validates
        # the already-probed writer invariant without opening the remote URI:
        # every logical timestamp must read back as Spark timestamp_ntz, whose
        # serverless Parquet representation was verified as INT64 microseconds.
        storage_types = {
            field.name: field.dataType.simpleString().lower()
            for field in df.schema.fields
        }
        timestamp_storage = {
            column["name"]: storage_types.get(column["name"])
            for column in expected
            if column["type"] == "timestamp"
        }
        gate(
            f"no_int96_{table}",
            all(storage_type == "timestamp_ntz" for storage_type in timestamp_storage.values()),
            {
                "mode": "verified_timestamp_ntz_writer_contract",
                "columns": timestamp_storage,
            },
        )
    gate(
        f"column_names_{table}",
        all(re.fullmatch(r"[a-z][a-z0-9_]*", name) for name in df.columns),
    )
gate("logical_column_count", total_columns == 267, total_columns)

for table, document in stats_docs.items():
    reread = json.loads(dbutils.fs.head(f"{staging}/stats/{table}.json", 10_000_000))
    gate(f"stats_table_{table}", reread["table"] == table and reread["basis"] == basis)
    gate(
        f"stats_columns_{table}",
        set(reread["columns"]).issubset(
            {column["name"] for column in tables_block[table]["columns"]}
        ),
    )

special_keys = {
    "episode_event": ["episode_id", "event_id"],
    "fact_relationship": [
        "domain_concept_id_1",
        "fact_id_1",
        "domain_concept_id_2",
        "fact_id_2",
        "relationship_concept_id",
    ],
    "concept_ancestor": ["ancestor_concept_id", "descendant_concept_id"],
    "concept_relationship": ["concept_id_1", "concept_id_2", "relationship_id"],
    "concept_synonym": ["concept_id", "concept_synonym_name", "language_concept_id"],
}
for table in logical:
    config = DATASET_CONFIG["tables"][table]
    key_columns = config.get("pk_columns") or special_keys.get(table) or [config["pk"]]
    df = staged[table]
    null_condition = None
    for column_name in key_columns:
        condition = F.col(_qcol(column_name)).isNull()
        null_condition = condition if null_condition is None else null_condition | condition
    gate(f"pk_non_null_{table}", df.where(null_condition).limit(1).count() == 0, key_columns)
    gate(
        f"pk_unique_{table}",
        df.groupBy(*[F.col(_qcol(name)) for name in key_columns])
        .count()
        .where(F.col("count") > 1)
        .limit(1)
        .count()
        == 0,
        key_columns,
    )

for table in logical:
    for foreign_key in DATASET_CONFIG["tables"][table].get("fks", []):
        parent_table, parent_column = foreign_key["references"].split(".", 1)
        child_column = foreign_key["column"]
        child_rows = staged[table].where(F.col(_qcol(child_column)).isNotNull())
        if parent_table == "concept":
            child_rows = child_rows.where(F.col(_qcol(child_column)) > 0)
        child = child_rows.select(F.col(_qcol(child_column)).alias("key"))
        parent = staged[parent_table].select(F.col(_qcol(parent_column)).alias("key"))
        missing = child.join(parent, "key", "left_anti").limit(1).count()
        gate(f"fk_{table}_{child_column}", missing == 0, foreign_key["references"])

gate(
    "infant_id_resolves_person",
    staged["infant"]
    .select(F.col("infant_id").alias("key"))
    .join(staged["person"].select(F.col("person_id").alias("key")), "key", "left_anti")
    .limit(1)
    .count()
    == 0,
)
gate(
    "observation_period_one_per_person",
    staged["observation_period"].count() == staged["person"].count()
    and staged["observation_period"].select("person_id").distinct().count()
    == staged["person"].count(),
)
gate(
    "episode_one_per_pregnancy",
    staged["episode"].count() == staged["pregnancy"].count()
    and staged["episode"].select("pregnancy_id").distinct().count()
    == staged["pregnancy"].count(),
)

event_targets = {
    "condition_occurrence": "condition_occurrence_id",
    "measurement": "measurement_id",
    "observation": "observation_id",
    "procedure_occurrence": "procedure_occurrence_id",
    "visit_occurrence": "visit_occurrence_id",
}
actual_sources = {row.source_table for row in staged["episode_event"].select("source_table").distinct().collect()}
gate("episode_event_source_types", actual_sources == set(event_targets), sorted(actual_sources))
for source_table, id_column in event_targets.items():
    child = staged["episode_event"].where(F.col("source_table") == source_table).select(
        F.col("event_id").alias("key")
    )
    parent = staged[source_table].select(F.col(id_column).alias("key"))
    gate(
        f"episode_event_resolves_{source_table}",
        child.join(parent, "key", "left_anti").limit(1).count() == 0,
    )

for endpoint in ("fact_id_1", "fact_id_2"):
    child = staged["fact_relationship"].select(F.col(endpoint).alias("key"))
    parent = staged["person"].select(F.col("person_id").alias("key"))
    gate(
        f"fact_relationship_{endpoint}_person",
        child.join(parent, "key", "left_anti").limit(1).count() == 0,
    )

concept_keys = staged["concept"].select(F.col("concept_id").alias("key"))
for table, columns in {
    "concept_relationship": ["concept_id_1", "concept_id_2"],
    "concept_ancestor": ["ancestor_concept_id", "descendant_concept_id"],
    "concept_synonym": ["concept_id", "language_concept_id"],
}.items():
    for column_name in columns:
        child = staged[table].where(F.col(column_name) > 0).select(F.col(column_name).alias("key"))
        gate(
            f"vocab_{table}_{column_name}",
            child.join(concept_keys, "key", "left_anti").limit(1).count() == 0,
        )

used_concepts = (
    staged["measurement"].select(F.col("measurement_concept_id").alias("key"))
    .union(staged["observation"].select(F.col("observation_concept_id").alias("key")))
    .union(staged["condition_occurrence"].select(F.col("condition_concept_id").alias("key")))
    .union(staged["visit_occurrence"].select(F.col("visit_concept_id").alias("key")))
    .where(F.col("key") > 0)
    .distinct()
)
gate(
    "used_concepts_resolve",
    used_concepts.join(concept_keys, "key", "left_anti").limit(1).count() == 0,
)
gate(
    "person_source_value_null",
    staged["person"].where(F.col("person_source_value").isNotNull()).limit(1).count() == 0,
)
gate("type_coercion_count", len(coercions) == 31, len(coercions))
gate(
    "concept_stats_labelled",
    any(
        top.get("label")
        for document in stats_docs.values()
        for column_name, column_stats in document["columns"].items()
        if _is_concept_column(column_name)
        for top in column_stats.get("top_values", [])
    ),
)
validation["status"] = "pass"
_mark("validation_done", gates=len(validation["gates"]))

In [0]:
profile_hash = hashlib.sha256(
    json.dumps(DATASET_CONFIG, sort_keys=True, default=str).encode("utf-8")
).hexdigest()
receipt = build_receipt(
    P["dataset_name"],
    run_at,
    run_id,
    "CC/Data Browser Publish",
    {
        **P,
        "suppression_threshold": P["k"],
        "seed": P["seed"],
        "target_file_mb": P["target_file_mb"],
        "run_at": run_at,
        "runtime_config": runtime_config,
    },
    snapshot,
    coercions,
    excluded_all,
    untagged_all,
    validation,
)
receipt.update(
    {
        "profile_sha256": profile_hash,
        "paths": {"staging": staging, "live": live, "archive": archive},
        "lossy_cast_failures": 0,
        "logical_table_count": len(logical),
        "logical_column_count": total_columns,
        "comparison_result_count": len(comparison_results),
        "comparison_basis": (
            comparisons_doc["basis"] if comparisons_doc else None
        ),
    }
)
dbutils.fs.put(
    f"{staging}/_receipt.json", json.dumps(receipt, indent=2, default=str), True
)

if P["dry_run"]:
    target = staging
else:
    for table in logical:
        current_real = _latest_history(rcat, rsch, table)["version"]
        current_synth = _latest_history(
            scat, ssch, f"{P['synthetic_prefix']}{table}"
        )["version"]
        assert current_real == snapshot[table]["real"]["delta_version"]
        assert current_synth == snapshot[table]["synth"]["delta_version"]
    had_live = True
    try:
        dbutils.fs.ls(live)
    except Exception:
        had_live = False
    if had_live:
        dbutils.fs.mv(live, archive, True)
    try:
        dbutils.fs.mv(staging, live, True)
        target = live
    except Exception:
        if had_live:
            dbutils.fs.mv(archive, live, True)
        raise

result = {
    "target": target,
    "run_id": run_id,
    "tables": len(logical),
    "columns": total_columns,
    "basis": basis,
    "type_coercions": len(coercions),
    "stats_files": len(stats_docs),
    "comparison_results": len(comparison_results),
    "comparison_basis": comparisons_doc["basis"] if comparisons_doc else None,
    "concept_labels_present": validation["gates"]["concept_stats_labelled"]["status"]
    == "pass",
    "sample_table": "pregnancy",
    "sample_rows": tables_block["pregnancy"]["rows"],
    "validation": validation["status"],
}
dbutils.fs.rm(diagnostic_path, False)
dbutils.fs.rm(failure_path, False)
dbutils.notebook.exit(json.dumps(result))